<a href="https://colab.research.google.com/github/Brown-Brayden/Problem-Set-1-Econ_3916_PS1/blob/main/Problem_Set_3_Econ_3916_PS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

---

A dashboard reports average revenue per customer in the West region as **\$1,240**, and a director has quoted it in a board deck. Finance ran its own query on the same warehouse and got **\$812**. This notebook rebuilds the warehouse from a known data-generating process and reproduces both numbers in DuckDB SQL. It finds the one difference between the two queries and writes the checks that would have caught it. It then takes the director's second question: did the West's account-management programme cause the West's higher revenue? Run it top to bottom with *Runtime › Run all*. Every random draw comes from the one seeded generator in Setup.

## Setup

In [1]:
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)
pd.set_option('display.max_colwidth', 140)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [2]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 5_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

# My choices inside those limits. 3,000 West customers (above the 2,000 minimum) steady the
# long-tailed order counts; the other 2,000 customers are split over three regions.
REGION_COUNTS = {"West": 3_000, "East": 800, "South": 700, "Central": 500}
FIRST_SIGNUP, LAST_SIGNUP = np.datetime64("2018-01-01"), np.datetime64("2025-12-31")
DATA_END = np.datetime64("2026-06-30")       # last day an order can be placed
assert sum(REGION_COUNTS.values()) == N_CUSTOMERS and REGION_COUNTS["West"] >= MIN_WEST

In [3]:
# Customers. Region is shuffled independently of everything else, so the only link
# between revenue and the join is the order count itself.
region = rng.permutation(np.repeat(list(REGION_COUNTS), list(REGION_COUNTS.values())))
n_orders = 1 + rng.negative_binomial(NB_N, NB_P, size=N_CUSTOMERS)
revenue = np.maximum(REV_PER_ORDER * n_orders + rng.normal(REV_MEAN, REV_SD, size=N_CUSTOMERS),
                     REV_FLOOR)
signup_days = rng.integers(0, (LAST_SIGNUP - FIRST_SIGNUP).astype(int) + 1, size=N_CUSTOMERS)
signup_date = FIRST_SIGNUP + signup_days.astype("timedelta64[D]")

customers_df = pd.DataFrame({
    "customer_id": np.arange(1, N_CUSTOMERS + 1),
    "region": region,
    "revenue": revenue.round(2),
    "signup_date": signup_date,
})

# Orders: one row per order, n_orders rows for each customer, dated between signup and DATA_END.
owner = np.repeat(np.arange(N_CUSTOMERS), n_orders)
days_live = (DATA_END - signup_date[owner]).astype(int)
order_date = signup_date[owner] + rng.integers(0, days_live + 1).astype("timedelta64[D]")
orders_df = pd.DataFrame({
    "order_id": np.arange(1, len(owner) + 1),
    "customer_id": customers_df["customer_id"].to_numpy()[owner],
    "order_date": order_date,
})
print(f"{len(customers_df):,} customers, {len(orders_df):,} orders")

5,000 customers, 34,061 orders


In [4]:
def load_warehouse(con, customers_df, orders_df):
    """Create customers and orders in a DuckDB connection from the two DataFrames."""
    con.register("customers_src", customers_df)
    con.register("orders_src", orders_df)
    con.execute("""
        CREATE TABLE customers (
            customer_id INTEGER PRIMARY KEY,
            region      VARCHAR,          -- no NOT NULL constraint, as in the real warehouse
            revenue     DOUBLE,
            signup_date DATE
        )""")
    con.execute("""
        INSERT INTO customers
        SELECT customer_id::INTEGER, region::VARCHAR, revenue::DOUBLE, signup_date::DATE
        FROM customers_src""")
    con.execute("""
        CREATE TABLE orders (
            order_id    INTEGER PRIMARY KEY,
            customer_id INTEGER NOT NULL,
            order_date  DATE
        )""")
    con.execute("""
        INSERT INTO orders
        SELECT order_id::INTEGER, customer_id::INTEGER, order_date::DATE
        FROM orders_src""")
    con.unregister("customers_src")
    con.unregister("orders_src")
    return con


load_warehouse(con, customers_df, orders_df)

In [5]:
display(con.sql("SHOW TABLES").df())
display(con.sql("DESCRIBE customers").df())
display(con.sql("DESCRIBE orders").df())

# One row per customer in customers, and every order belongs to a known customer.
display(con.sql("""
SELECT (SELECT COUNT(*) FROM customers)                    AS customer_rows,
       (SELECT COUNT(DISTINCT customer_id) FROM customers) AS distinct_customer_ids,
       (SELECT COUNT(*) FROM orders)                       AS order_rows,
       (SELECT COUNT(DISTINCT customer_id) FROM orders)    AS customers_with_orders,
       (SELECT COUNT(*) FROM orders
         WHERE customer_id NOT IN (SELECT customer_id FROM customers)) AS orphan_orders
""").df())

# Order counts have a mean near 7 and a long right tail, and revenue rises with them.
display(con.sql("""
WITH per_customer AS (
    SELECT customer_id, COUNT(*) AS n_orders
    FROM orders
    GROUP BY customer_id
)
SELECT COALESCE(c.region, 'All regions')  AS region,
       COUNT(*)                           AS customers,
       AVG(p.n_orders)                    AS mean_orders,
       MEDIAN(p.n_orders)                 AS median_orders,
       QUANTILE_CONT(p.n_orders, 0.99)    AS p99_orders,
       MAX(p.n_orders)                    AS max_orders,
       AVG(c.revenue)                     AS mean_revenue,
       CORR(c.revenue, p.n_orders)        AS corr_revenue_orders
FROM customers AS c
JOIN per_customer AS p ON p.customer_id = c.customer_id
GROUP BY ROLLUP (c.region)
ORDER BY customers DESC
""").df())

,name
0,customers
1,orders


,column_name,column_type,null,key,default,extra
0,customer_id,INTEGER,NO,PRI,None,None
1,region,VARCHAR,YES,None,None,None
2,revenue,DOUBLE,YES,None,None,None
3,signup_date,DATE,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,order_id,INTEGER,NO,PRI,None,None
1,customer_id,INTEGER,NO,None,None,None
2,order_date,DATE,YES,None,None,None


,customer_rows,distinct_customer_ids,order_rows,customers_with_orders,orphan_orders
0,5000,5000,34061,5000,0


,region,customers,mean_orders,median_orders,p99_orders,max_orders,mean_revenue,corr_revenue_orders
0,All regions,5000,6.812,4.0,31.01,57,807.549,0.939
1,West,3000,6.539,4.0,30.00,54,789.201,0.933
2,East,800,7.350,5.0,37.01,55,851.502,0.953
3,South,700,6.763,4.0,30.01,57,799.387,0.939
4,Central,500,7.658,5.0,30.03,42,858.747,0.945


**What this builds.** The warehouse has two tables, both drawn from the one seeded generator in Setup (`default_rng(42)`). `customers` has 5,000 rows, one per `customer_id` (the primary key), and 3,000 of them are in the West. `orders` has 34,061 rows, one per order, and every customer has at least one. Order counts follow 1 + NegBin(0.8, 0.12): mean 6.81, median 4, 99th percentile 31, maximum 57. Most customers place a handful of orders and a few place dozens. Revenue is 60 × orders + N(400, 150), floored at 50, so it is tied to order count by construction (correlation 0.94). Region is assigned at random, independently of everything else. Any difference between the two queries must therefore come from the join.

### Step 1.2: Write both queries (7 pts)

In [6]:
Q_FINANCE = """
SELECT AVG(c.revenue) AS avg_revenue,
       COUNT(*)       AS n_rows
FROM customers AS c
WHERE c.region = 'West'
""".strip()

Q_DASHBOARD = """
SELECT AVG(c.revenue) AS avg_revenue,
       COUNT(*)       AS n_rows
FROM customers AS c
JOIN orders AS o
  ON o.customer_id = c.customer_id
WHERE c.region = 'West'
""".strip()

finance_avg, finance_rows = con.sql(Q_FINANCE).fetchone()
dashboard_avg, dashboard_rows = con.sql(Q_DASHBOARD).fetchone()
ratio = dashboard_avg / finance_avg

display(pd.DataFrame({
    "query": ["Finance: customers only", "Dashboard: customers JOIN orders"],
    "avg_revenue": [finance_avg, dashboard_avg],
    "rows_averaged": [finance_rows, dashboard_rows],
}))
print(f"dashboard / finance = {ratio:.3f}")

,query,avg_revenue,rows_averaged
0,Finance: customers only,789.201,3000
1,Dashboard: customers JOIN orders,1182.398,19618


dashboard / finance = 1.498


> **🛑 Verification checkpoint.** On West-region customers you should get about
> **\$812** from the un-joined query (accept \$780 to \$845) and about **\$1,240**
> from the joined one (accept \$1,160 to \$1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

In [7]:
checkpoint = {
    "un-joined average in $780-$845": 780 <= finance_avg <= 845,
    "joined average in $1,160-$1,320": 1_160 <= dashboard_avg <= 1_320,
    "ratio in 1.45-1.60": 1.45 <= ratio <= 1.60,
}
for name, ok in checkpoint.items():
    print("PASS" if ok else "FAIL", " ", name)
assert all(checkpoint.values()), "Checkpoint not met: change the seed in Setup and rerun from the top."

PASS   un-joined average in $780-$845
PASS   joined average in $1,160-$1,320
PASS   ratio in 1.45-1.60


**Checkpoint met.** The Finance query returns **\$789.20** over 3,000 rows. The dashboard query returns **\$1,182.40** over 19,618 rows. The ratio is 1.498. All three are inside their bands, and the cell above prints three PASS lines. Both numbers sit a little below the deck's \$812 and \$1,240 for one shared reason: West customers in this draw average 6.54 orders, against the design's 6.87. The bands exist to allow for that kind of draw.

In [8]:
# The joined AVG is a per-customer average weighted by order count. Rebuild it that way.
display(con.sql("""
WITH per_customer AS (
    SELECT customer_id, COUNT(*) AS n_orders
    FROM orders
    GROUP BY customer_id
)
SELECT AVG(c.revenue)                                AS each_customer_once,
       SUM(c.revenue * p.n_orders) / SUM(p.n_orders) AS each_customer_once_per_order,
       COUNT(*)                                      AS west_customers,
       SUM(p.n_orders)                               AS west_orders
FROM customers AS c
JOIN per_customer AS p ON p.customer_id = c.customer_id
WHERE c.region = 'West'
""").df())

# Who the join repeats: West customers in tenths by order count, most orders first.
display(con.sql("""
WITH per_customer AS (
    SELECT customer_id, COUNT(*) AS n_orders
    FROM orders
    GROUP BY customer_id
),
west AS (
    SELECT c.revenue, p.n_orders,
           NTILE(10) OVER (ORDER BY p.n_orders DESC, c.customer_id) AS order_decile
    FROM customers AS c
    JOIN per_customer AS p ON p.customer_id = c.customer_id
    WHERE c.region = 'West'
)
SELECT order_decile,
       COUNT(*)                                            AS customers,
       AVG(n_orders)                                       AS avg_orders,
       AVG(revenue)                                        AS avg_revenue,
       100.0 * COUNT(*) / SUM(COUNT(*)) OVER ()            AS pct_of_customers,
       100.0 * SUM(n_orders) / SUM(SUM(n_orders)) OVER ()  AS pct_of_joined_rows
FROM west
GROUP BY order_decile
ORDER BY order_decile
""").df())

,each_customer_once,each_customer_once_per_order,west_customers,west_orders
0,789.201,1182.398,3000,19618.0


,order_decile,customers,avg_orders,avg_revenue,pct_of_customers,pct_of_joined_rows
0,1,300,22.023,1719.306,10.0,33.678
1,2,300,12.300,1140.929,10.0,18.809
2,3,300,8.820,916.024,10.0,13.488
3,4,300,6.517,792.127,10.0,9.965
4,5,300,5.033,704.239,10.0,7.697
5,6,300,3.790,626.618,10.0,5.796
6,7,300,2.837,551.924,10.0,4.338
7,8,300,2.000,499.237,10.0,3.058
8,9,300,1.073,475.043,10.0,1.641
9,10,300,1.000,466.559,10.0,1.529


**The Finance number is the correct one (\$812 in the deck, \$789.20 in this rebuild), because the unit of observation is the customer.** The metric is *average revenue per customer*, so each West customer should count once. `revenue` is a customer-level attribute: it lives in `customers`, which has one row per `customer_id`. The Finance query averages it over exactly that, 3,000 rows for 3,000 West customers.

The dashboard query joins `orders` before it averages, and that changes the unit of observation from the customer to the order. The result has 19,618 rows, one per order, and each row carries a copy of its customer's revenue. `AVG` then counts every customer once per order: a customer with 40 orders counts 40 times, and a customer with one order counts once. The first table above shows that the dashboard number is exactly Σ(revenue × orders) / Σ orders = \$1,182.40. The weighting accounts for the entire gap.

Weighting moves the average only because revenue rises with order count. The second table shows that the top tenth of West customers by order count is 10% of customers but 33.7% of the rows the dashboard averages. Those are also the highest-revenue customers: they average 22 orders and \$1,719 of revenue each.

The dashboard number (\$1,240 in the deck, \$1,182.40 here) is correct arithmetic, but it answers a different question: what is the revenue of the customer behind a randomly chosen order? Nobody asked that. Neither query is the tidier one. One counts customers, the other counts orders, and the metric's name says customers.

### Step 1.3: The rule (10 pts)

**The rule.** A metric about customers must be computed on rows that hold one row per customer. A join may add columns to those rows, but it must not add rows. Before any join, the two counts below are equal, because `customer_id` is the primary key of `customers`. A join that fans out pulls them apart. In SQL, the check runs on the metric's own FROM, JOIN and WHERE clauses:

```sql
SELECT COUNT(*)                                  AS rows_in,
       COUNT(DISTINCT c.customer_id)             AS customers_in,
       COUNT(*) = COUNT(DISTINCT c.customer_id)  AS one_row_per_customer
FROM customers AS c
JOIN orders AS o ON o.customer_id = c.customer_id
WHERE c.region = 'West'
```

`check_fanout` below runs this check on any query of that shape. It keeps the query's FROM, JOIN and WHERE clauses exactly as written, and it warns when the join has changed the row count.

In [9]:
class FanOutWarning(UserWarning):
    """A join changed the number of rows per unit of observation."""


def check_fanout(con, sql, key="c.customer_id"):
    """Warn if the rows a query aggregates hold more than one row per `key`.

    customer_id is the primary key of customers, so before any join the row source has
    exactly one row per customer and COUNT(*) equals COUNT(DISTINCT key). If a join breaks
    that equality, it changed the row count, and an AVG over those rows weights each
    customer by how many times the join repeated them.

    Handles one SELECT ... FROM ... [WHERE ...] block without GROUP BY, which covers both
    Phase 1 queries. It swaps the SELECT list for the two counts and leaves the FROM, JOIN
    and WHERE clauses exactly as written.
    """
    match = re.match(r"\s*SELECT\b.*?\bFROM\b(.*)", sql, flags=re.IGNORECASE | re.DOTALL)
    if match is None:
        raise ValueError("expected a query of the form SELECT ... FROM ...")
    n_rows, n_units = con.sql(
        f"SELECT COUNT(*), COUNT(DISTINCT {key}) FROM {match.group(1)}"
    ).fetchone()
    result = {"rows": n_rows, "units": n_units,
              "rows_per_unit": n_rows / n_units, "fanout": n_rows != n_units}
    if result["fanout"]:
        warnings.warn(
            f"join changed the row count: {n_units:,} customers became {n_rows:,} rows "
            f"({n_rows / n_units:.2f} per customer), so an AVG over these rows is weighted "
            "by how many times each customer was repeated",
            FanOutWarning, stacklevel=2)
    return result

In [10]:
# A third query: the right way to bring orders in. Roll them up to one row per customer, then join.
Q_FIXED = """
SELECT AVG(c.revenue)  AS avg_revenue,
       AVG(o.n_orders) AS avg_orders,
       COUNT(*)        AS n_rows
FROM customers AS c
JOIN (SELECT customer_id, COUNT(*) AS n_orders
      FROM orders
      GROUP BY customer_id) AS o
  ON o.customer_id = c.customer_id
WHERE c.region = 'West'
""".strip()

demo = []
for label, sql in [("Dashboard (the $1,240)", Q_DASHBOARD),
                   ("Finance (the $812)", Q_FINANCE),
                   ("Orders rolled up first, then joined", Q_FIXED)]:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = check_fanout(con, sql)
    demo.append({"query": label, **result, "warned": bool(caught),
                 "warning": str(caught[0].message) if caught else ""})
demo = pd.DataFrame(demo)
display(demo)
assert demo["warned"].tolist() == [True, False, False]

display(con.sql(Q_FIXED).df())   # still the per-customer number, with order counts alongside
check_fanout(con, Q_DASHBOARD);  # the warning as an analyst would see it

,query,rows,units,rows_per_unit,fanout,warned,warning
0,"Dashboard (the $1,240)",19618,3000,6.539,True,True,"join changed the row count: 3,000 customers became 19,618 rows (6.54 per customer), so an AVG over these rows is weighted by how many ti..."
1,Finance (the $812),3000,3000,1.000,False,False,
2,"Orders rolled up first, then joined",3000,3000,1.000,False,False,


,avg_revenue,avg_orders,n_rows
0,789.201,6.539,3000


/tmp/ipykernel_2767/701140262.py:28: FanOutWarning: join changed the row count: 3,000 customers became 19,618 rows (6.54 per customer), so an AVG over these rows is weighted by how many times each customer was repeated
  check_fanout(con, Q_DASHBOARD);  # the warning as an analyst would see it


**It fires on the bad query and stays silent on the good one.** On the dashboard query it warns that 3,000 West customers became 19,618 rows, 6.54 per customer. On the Finance query, 3,000 rows hold 3,000 customers, so it stays silent. It is also silent on a third query that brings orders in the right way: it rolls `orders` up to one row per customer and joins that. That query still returns \$789.20, with the average order count (6.54) alongside. So the rule is not "never join". It is "a join may add columns, not rows". The check belongs in the dashboard's build step, where a warning stops the publish before a number reaches a slide.

*Limits.* It reads a single `SELECT … FROM … WHERE` block. A query that starts with `WITH` raises an error instead of passing. A query with `GROUP BY` needs one check per group, and the function only reads the first. It also has to be told the unit (`key`). Phase 4 asks an AI for a more general reviewer and tests it on these same queries.

## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [11]:
# Region became mandatory on 1 Jan 2020. Most customers onboarded before then were never
# backfilled, so the NULLs sit on the oldest accounts instead of falling at random.
MANDATORY_FROM = np.datetime64("2020-01-01")
P_NOT_BACKFILLED = 0.60   # 2018-19 signups are a quarter of customers, so about 15% of rows go NULL

legacy = customers_df["signup_date"].to_numpy() < MANDATORY_FROM
goes_null = legacy & (rng.random(N_CUSTOMERS) < P_NOT_BACKFILLED)
null_ids = customers_df.loc[goes_null, ["customer_id"]]

con.register("null_ids", null_ids)
con.execute("""
UPDATE customers
SET region = NULL
WHERE customer_id IN (SELECT customer_id FROM null_ids)
""")
con.unregister("null_ids")

# customers_df still holds every customer's true region. Keep it queryable for comparison.
con.register("customers_truth", customers_df)

display(con.sql("""
SELECT COUNT(*)                                       AS customers,
       COUNT(region)                                  AS region_recorded,
       COUNT(*) - COUNT(region)                       AS region_null,
       100.0 * (COUNT(*) - COUNT(region)) / COUNT(*)  AS pct_null
FROM customers
""").df())

# Who went missing, measured against the truth we kept.
display(con.sql("""
SELECT CASE WHEN c.region IS NULL THEN 'region NULL' ELSE 'region recorded' END AS status,
       COUNT(*)                                                   AS customers,
       MIN(c.signup_date)                                         AS first_signup,
       MAX(c.signup_date)                                         AS last_signup,
       AVG(c.revenue)                                             AS avg_revenue,
       100.0 * AVG(CASE WHEN t.region = 'West' THEN 1 ELSE 0 END) AS pct_truly_west
FROM customers AS c
JOIN customers_truth AS t ON t.customer_id = c.customer_id
GROUP BY status
ORDER BY status
""").df())

,customers,region_recorded,region_null,pct_null
0,5000,4269,731,14.62


,status,customers,first_signup,last_signup,avg_revenue,pct_truly_west
0,region NULL,731,2018-01-02,2019-12-31,826.284,60.328
1,region recorded,4269,2018-01-01,2025-12-29,804.341,59.944


**What changed.** 731 of 5,000 customers (14.6%) now have region NULL. The rule behind them is that region became a required field on 1 January 2020, and 60% of the accounts opened before then were never backfilled. So the unknowns are not a random sample. Every one of them signed up between January 2018 and December 2019, which makes them systematically older than the rest. Their average revenue is close to everyone else's (\$826 against \$804, a gap within sampling noise), because revenue in this simulation depends only on order count. In a real warehouse there is no reason to expect that. We kept the true regions, so we also know that 60.3% of the unknowns are actually West customers.

### Step 2.2: Lose them, then find them (15 pts)

In [12]:
Q_NOT_WEST_NAIVE = """
SELECT COUNT(*)     AS n_customers,
       AVG(revenue) AS avg_revenue
FROM customers
WHERE region != 'West'
""".strip()

Q_NOT_WEST_MEANT = """
SELECT COUNT(*)     AS n_customers,
       AVG(revenue) AS avg_revenue
FROM customers
WHERE region IS DISTINCT FROM 'West'
""".strip()

naive_n, naive_avg = con.sql(Q_NOT_WEST_NAIVE).fetchone()
meant_n, meant_avg = con.sql(Q_NOT_WEST_MEANT).fetchone()
west_n, total_n = con.sql(
    "SELECT COUNT(*) FILTER (WHERE region = 'West'), COUNT(*) FROM customers"
).fetchone()

display(pd.DataFrame({
    "query": ["WHERE region != 'West'", "WHERE region IS DISTINCT FROM 'West'"],
    "rows": [naive_n, meant_n],
    "avg_revenue": [naive_avg, meant_avg],
}))
print(f"rows the first query silently drops: {meant_n - naive_n:,}")
print(f"region = 'West' ({west_n:,}) + region != 'West' ({naive_n:,}) = {west_n + naive_n:,} "
      f"of {total_n:,} rows")

,query,rows,avg_revenue
0,WHERE region != 'West',1710,835.318
1,WHERE region IS DISTINCT FROM 'West',2441,832.613


rows the first query silently drops: 731
region = 'West' (2,559) + region != 'West' (1,710) = 4,269 of 5,000 rows


In [13]:
# Three-valued logic: a comparison with NULL is neither TRUE nor FALSE. It is NULL (unknown).
display(con.sql("""
SELECT CAST(NULL AS VARCHAR) =  'West'               AS "NULL = 'West'",
       CAST(NULL AS VARCHAR) != 'West'               AS "NULL != 'West'",
       NOT (CAST(NULL AS VARCHAR) = 'West')          AS "NOT (NULL = 'West')",
       CAST(NULL AS VARCHAR) IS DISTINCT FROM 'West' AS "NULL IS DISTINCT FROM 'West'"
""").df())

# Every regional filter loses rows, not just "not West": true region against what is recorded.
display(con.sql("""
SELECT t.region                   AS true_region,
       COUNT(*)                   AS customers,
       COUNT(c.region)            AS region_recorded,
       COUNT(*) - COUNT(c.region) AS dropped_by_any_region_filter
FROM customers_truth AS t
JOIN customers AS c ON c.customer_id = t.customer_id
GROUP BY t.region
ORDER BY customers DESC
""").df())

# The honest regional table: Unknown is a region of its own.
display(con.sql("""
SELECT COALESCE(region, 'Unknown')              AS region,
       COUNT(*)                                 AS customers,
       100.0 * COUNT(*) / SUM(COUNT(*)) OVER () AS pct_of_customers,
       AVG(revenue)                             AS avg_revenue
FROM customers
GROUP BY COALESCE(region, 'Unknown')
ORDER BY customers DESC
""").df())

,NULL = 'West',NULL != 'West',NOT (NULL = 'West'),NULL IS DISTINCT FROM 'West'
0,<NA>,<NA>,<NA>,True


,true_region,customers,region_recorded,dropped_by_any_region_filter
0,West,3000,2559,441
1,East,800,685,115
2,South,700,589,111
3,Central,500,436,64


,region,customers,pct_of_customers,avg_revenue
0,West,2559,51.18,783.642
1,Unknown,731,14.62,826.284
2,East,685,13.70,850.576
3,South,589,11.78,799.257
4,Central,436,8.72,860.062


**The difference.** `WHERE region != 'West'` returns 1,710 rows. The query the analyst meant returns 2,441. The first one silently drops **731 customers**, every one whose region is NULL. West plus "not West" no longer adds up to the table: 2,559 + 1,710 = 4,269 of 5,000 rows. The two averages barely differ here (\$835 against \$833), only because these unknowns happen to resemble everyone else on revenue. The count is what breaks, and nothing in the output says so.

**The mechanism, in one sentence.** SQL uses three-valued logic, so `NULL != 'West'` evaluates to UNKNOWN rather than TRUE, and because `WHERE` keeps only rows whose condition is TRUE, every NULL-region row is discarded with no error and no warning.

**What the analyst meant.** They meant "everyone except the West". A customer with no recorded region is not known to be West, so that customer belongs in the answer. `region IS DISTINCT FROM 'West'` treats NULL as a value that differs from 'West', so it keeps those rows. Writing `region != 'West' OR region IS NULL` does the same thing. We know from the simulation that 60.3% of the unknowns are really West customers, though. So the most honest report keeps them as a separate *Unknown* line, as in the last table, rather than folding them into either side. The second table shows that the loss is not confined to "not West". Every filter on region drops 13% to 16% of each region's true customers, including 441 West customers that `WHERE region = 'West'` now misses.

**The consequence for every regional comparison.** Suppose the missing-region customers are systematically different: newer, smaller, more likely to churn. Then the first query does not just lose rows. It biases every regional number in the company in the same direction. Each filter or `GROUP BY` on region keeps the customers whose records are complete and drops the small, at-risk accounts. Every regional average revenue then comes out too high, every regional churn rate too low and every regional customer count too small. The bias is not equal across regions either. A region that onboarded more customers before the field was required loses more of them and looks better by more. A gap such as "West outperforms East" can then come from who is missing rather than from anything the regions did. Regional totals also stop adding up to the company total, so the missing revenue belongs to no one. This is the same problem as in Phase 3: the analysis runs on a sample selected on something related to the outcome, and more rows do not fix it. The fix is a reporting rule. Every regional cut shows Unknown as its own line, with its share, and `!=` on a nullable column fails review.

### Step 2.3: Aggregate without collapsing (10 pts)

In [14]:
Q_WINDOW = """
SELECT customer_id,
       region,
       revenue,
       AVG(revenue) OVER w           AS region_avg_revenue,
       revenue - AVG(revenue) OVER w AS diff_from_region_avg,
       CASE WHEN revenue > AVG(revenue) OVER w THEN 'above'
            ELSE 'at or below' END   AS vs_region_avg
FROM customers
WINDOW w AS (PARTITION BY region)
ORDER BY region, customer_id
""".strip()

by_customer = con.sql(Q_WINDOW).df()
rows_in = con.sql("SELECT COUNT(*) FROM customers").fetchone()[0]
print(f"rows in customers: {rows_in:,} | rows out of the window query: {len(by_customer):,}")
assert len(by_customer) == rows_in
display(by_customer.head(8))

# Summary of the output, by region (NULL region shown as NaN).
display(by_customer.groupby("region", dropna=False).agg(
    customers=("customer_id", "size"),
    region_avg_revenue=("region_avg_revenue", "first"),
    above_avg=("vs_region_avg", lambda s: int((s == "above").sum())),
))

rows in customers: 5,000 | rows out of the window query: 5,000


,customer_id,region,revenue,region_avg_revenue,diff_from_region_avg,vs_region_avg
0,20,Central,457.79,860.062,-402.272,at or below
1,23,Central,842.77,860.062,-17.292,at or below
2,35,Central,1138.28,860.062,278.218,above
3,38,Central,1655.03,860.062,794.968,above
4,50,Central,844.13,860.062,-15.932,at or below
5,58,Central,1191.33,860.062,331.268,above
6,66,Central,432.97,860.062,-427.092,at or below
7,81,Central,898.92,860.062,38.858,above


,customers,region_avg_revenue,above_avg
region,,,
Central,436,860.062,167
East,685,850.576,252
South,589,799.257,230
West,2559,783.642,997
NaN,731,826.284,274


**One query, no join, no subquery, and the row count does not change.** `AVG(revenue) OVER (PARTITION BY region)` computes each region's average and writes it back onto every row in that region. `GROUP BY` would collapse the region to one row. The named window `w` lets the average, the gap from it and the above/below flag share one definition. 5,000 rows go in and 5,000 rows come out, and the `assert` checks that. Each customer's row now shows their revenue next to their region's average, which is what the director asked for. The 731 customers with no region are not dropped. `PARTITION BY` puts all the NULLs in one partition of their own, so they are compared with the average of the other unknowns (\$826.28). That is the only fair comparison available for them.

## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [15]:
# ---- A world where we know the truth (Step 3.1) ----
TRUE_EFFECT = 40.0     # the programme adds $40 of revenue per customer: small, by design
N_P3 = 10_000


def simulate_programme(n, rng):
    """One simulated customer base. account_size confounds region and revenue;
    support_tickets is a collider, caused by both the programme and revenue."""
    # Existing account size: last year's spend in $, fixed before the programme started.
    account_size = rng.lognormal(mean=np.log(600), sigma=0.6, size=n)
    # Larger accounts were more likely to be assigned to the West book of business.
    p_west = 1 / (1 + np.exp(-(-0.5 + 1.5 * np.log(account_size / 600))))
    programme = (rng.random(n) < p_west).astype(int)   # the programme ran in the West only
    # Revenue: account size, plus the programme's true effect, plus noise.
    revenue = 150 + account_size + TRUE_EFFECT * programme + rng.normal(0, 120, size=n)
    # Support tickets (Step 3.2): account managers log issues for their customers, and bigger
    # spenders raise more tickets. Nothing else drives them.
    tickets = rng.poisson(np.clip(1 + 6 * programme + revenue / 100, 0.1, None))
    return pd.DataFrame({
        "region": np.where(programme == 1, "West", "Other"),
        "programme": programme,
        "account_size": account_size,
        "revenue": revenue,
        "support_tickets": tickets,
    })


p3 = simulate_programme(N_P3, rng)
con.register("programme_eval", p3)

display(con.sql("""
SELECT region,
       COUNT(*)             AS customers,
       AVG(account_size)    AS avg_account_size,
       AVG(revenue)         AS avg_revenue,
       AVG(support_tickets) AS avg_tickets
FROM programme_eval
GROUP BY region
ORDER BY region DESC
""").df())

,region,customers,avg_account_size,avg_revenue,avg_tickets
0,West,3976,931.385,1119.346,18.241
1,Other,6024,584.013,732.974,8.318


In [16]:
def programme_effect(formula, data=p3):
    """OLS coefficient on programme, with a heteroskedasticity-robust 95% CI."""
    fit = smf.ols(formula, data=data).fit(cov_type="HC1")
    low, high = fit.conf_int().loc["programme"]
    return {"estimate": fit.params["programme"], "ci_low": low, "ci_high": high}


def against_truth(rows):
    out = pd.DataFrame(rows)
    out["truth"] = TRUE_EFFECT
    out["error"] = out["estimate"] - TRUE_EFFECT
    out["ci_covers_truth"] = (out["ci_low"] <= TRUE_EFFECT) & (TRUE_EFFECT <= out["ci_high"])
    return out


MODELS = {
    "naive: West minus everyone else": "revenue ~ programme",
    "adjusted for account size": "revenue ~ programme + account_size",
}
p31 = against_truth([{"model": name, **programme_effect(f)} for name, f in MODELS.items()])
display(p31)

,model,estimate,ci_low,ci_high,truth,error,ci_covers_truth
0,naive: West minus everyone else,386.371,365.881,406.862,40.0,346.371,False
1,adjusted for account size,38.271,33.136,43.407,40.0,-1.729,True


**Naive against adjusted, both against the truth.** In this world the programme's true effect is **+\$40 per customer**. The naive comparison, West's average revenue minus everyone else's, says **\$386.37** (95% CI \$365.88 to \$406.86), nearly ten times the truth. Almost all of that is account size. West was given the bigger accounts (average account size \$931 against \$584), and bigger accounts spend more whether or not they have an account manager. The adjusted estimate compares customers of the same account size, by regressing revenue on the programme and account size. It gives **\$38.27** (95% CI \$33.14 to \$43.41), which covers the truth.

That adjustment works here only because I built the world. Account size is the only confounder, it is measured, it was fixed before the programme started, and its effect on revenue is a straight line. With the company's real data, none of that is known. That is why the director's second question is harder than the first. The region that got the programme was not chosen at random, so the raw gap cannot be read as the programme's effect. An adjusted estimate is only as good as the list of things it adjusts for, and the list can be wrong in both directions, as the next step shows.

### Step 3.2: The control that ruins it (13 pts)

In [17]:
MODELS["adjusted for account size + support tickets"] = (
    "revenue ~ programme + account_size + support_tickets")
p32 = against_truth([{"model": name, **programme_effect(f)} for name, f in MODELS.items()])
display(p32)

# The mechanism: at the same ticket count, programme customers spend less than everyone else.
bands = p3.assign(ticket_band=pd.cut(p3["support_tickets"], [-1, 6, 10, 14, 19, 1_000],
                                     labels=["0-6", "7-10", "11-14", "15-19", "20+"]))
display(bands.pivot_table(index="ticket_band", columns="region", values="revenue",
                          aggfunc=["mean", "size"], observed=True))

,model,estimate,ci_low,ci_high,truth,error,ci_covers_truth
0,naive: West minus everyone else,386.371,365.881,406.862,40.0,346.371,False
1,adjusted for account size,38.271,33.136,43.407,40.0,-1.729,True
2,adjusted for account size + support tickets,-30.483,-36.812,-24.155,40.0,-70.483,False


mean            size      
region          Other      West Other  West
ticket_band                                
0-6           496.820   504.069  2379    48
7-10          713.990   641.726  2054   373
11-14         942.873   783.645  1008   897
15-19        1237.841  1002.491   427  1262
20+          1846.098  1589.459   156  1396

In [18]:
# Not a lucky draw: rebuild the world 500 times and see where each estimate lands.
def programme_coef(d, controls):
    X = np.column_stack([np.ones(len(d)), d["programme"], *(d[c] for c in controls)])
    return np.linalg.lstsq(X, d["revenue"].to_numpy(), rcond=None)[0][1]


SPECS = {"naive": [], "adjusted": ["account_size"],
         "adjusted + support tickets": ["account_size", "support_tickets"]}
mc = pd.DataFrame([
    {name: programme_coef(d, controls) for name, controls in SPECS.items()}
    for d in (simulate_programme(5_000, rng) for _ in range(500))
])
mc_summary = mc.agg(["mean", "std", "min", "max"]).T
mc_summary["truth"] = TRUE_EFFECT
display(mc_summary)

,mean,std,min,max,truth
naive,381.886,15.043,340.104,428.539,40.0
adjusted,40.244,3.645,29.072,50.238,40.0
adjusted + support tickets,-27.632,4.446,-40.704,-16.606,40.0


**What the extra control does.** Adding support tickets moves the estimate from \$38.27, which was right, to **−\$30.48** (95% CI −\$36.81 to −\$24.16). The programme now appears to *lower* revenue by about \$30 per customer, when in fact it raises revenue by \$40. The interval is nowhere near the truth, so this is not noise. Across 500 rebuilt worlds the ticket-adjusted estimate ranges from −\$40.70 to −\$16.61 and is negative every time, while the account-size estimate centres on \$40.24. The banded table shows how it happens. Nearly all West customers (99%) have 7 or more tickets, and in every band from 7 up, West customers have lower revenue than everyone else: at 11 to 14 tickets, for example, \$784 against \$943. A West customer gets about six extra tickets from their account manager, so they reach any given ticket count with less spending. Holding tickets fixed therefore compares programme customers with richer non-programme customers.

**The paragraph I would send the director:**

> Support tickets sound like a fair measure of engagement, but in our data the programme itself produces them: account managers log issues on their customers' behalf, so West customers raise more tickets at any level of spending. Bigger spenders also raise more tickets. Now compare two customers with the same ticket count. The West customer got part of that count from their account manager. The non-West customer reached the same count only by spending more. "Same number of tickets" therefore lines up West customers against richer customers elsewhere, and the programme looks worse than it is. In a test where we know the programme adds \$40 per customer, controlling for account size alone found \$38. Adding tickets turned that into −\$30, the wrong sign. The rule I would apply from now on: only control for things that were already fixed before the programme started, such as last year's account size. Anything the programme could have changed, such as tickets, logins or meetings, carries part of the programme's effect, and holding it constant distorts the answer instead of sharpening it.

## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

The reviewer below was written by Claude Opus 5.5 from the brief's P.R.I.M.E. prompt. The full exchange, unedited, is in the **AI Transcript** at the end of this notebook.

### Task 4.1: The query reviewer (12 pts)

In [19]:
# ---- The model's code (Cell 1 of its reply), pasted exactly as it returned it; see the AI Transcript. ----
# ---- Every line I changed or added is marked "# CHANGED:" with the reason.                            ----
import json
import re
from dataclasses import dataclass, field, asdict

import duckdb
import pandas as pd

JOIN_ROW_COUNT_CHANGE = "JOIN_ROW_COUNT_CHANGE"
NULLABLE_FILTER_DROPS_UNKNOWN = "NULLABLE_FILTER_DROPS_UNKNOWN"
AGGREGATE_OVER_FANOUT = "AGGREGATE_OVER_FANOUT"
REVIEW_INCOMPLETE = "REVIEW_INCOMPLETE"   # something wasn't analysed; never a silent pass


@dataclass
class QueryWarning:
    code: str        # one of the constants above; assert on this in tests
    severity: str    # "high" | "medium" | "low" | "info"
    message: str     # one-line explanation
    details: dict = field(default_factory=dict)   # the numbers behind the message


def warnings_frame(warnings):
    """One row per warning, for display() in Colab."""
    return pd.DataFrame([asdict(w) for w in warnings],
                        columns=["code", "severity", "message", "details"])


# Aggregates whose answer doesn't change when input rows are repeated.
_DUP_INSENSITIVE = {"min", "max", "any_value", "arbitrary", "first", "last",
                    "arg_min", "arg_max", "argmin", "argmax", "min_by", "max_by",
                    "bool_and", "bool_or", "bit_and", "bit_or", "approx_count_distinct"}
# Constructs showing the author decided what happens to NULLs.
_NULL_AWARE_TYPES = {"OPERATOR_IS_NULL", "OPERATOR_IS_NOT_NULL", "OPERATOR_COALESCE",
                     "COMPARE_DISTINCT_FROM", "COMPARE_NOT_DISTINCT_FROM"}
_NULL_AWARE_FUNCS = {"coalesce", "ifnull", "nvl"}


# ---------- parsing helpers (DuckDB's parser and deparser) ----------

def _parse(con, sql):
    return json.loads(con.execute("SELECT json_serialize_sql(?)", [sql]).fetchone()[0])


def _render(con, expr):
    """Expression AST -> SQL text, via DuckDB's deparser."""
    shell = _parse(con, "SELECT 1")
    shell["statements"][0]["node"]["select_list"] = [dict(expr, alias="")]
    text = con.execute("SELECT json_deserialize_sql(?::JSON)",
                       [json.dumps(shell)]).fetchone()[0]
    return re.sub(r"^\s*SELECT\s+", "", text.strip().rstrip(";"), flags=re.IGNORECASE)


def _q(name):
    return '"' + str(name).replace('"', '""') + '"'


def _exprs(tree):
    """Every expression node under tree; does not enter subqueries (different scope)."""
    if isinstance(tree, list):
        for item in tree:
            yield from _exprs(item)
    elif isinstance(tree, dict):
        if "class" in tree:
            yield tree
            if tree["class"] == "SUBQUERY":
                return
        for value in tree.values():
            if isinstance(value, (dict, list)):
                yield from _exprs(value)


def _columns(tree):
    return [e for e in _exprs(tree) if e["class"] == "COLUMN_REF"]


def _conjuncts(expr):
    if not expr:
        return []
    if expr.get("type") == "CONJUNCTION_AND":
        return [c for child in expr.get("children", []) for c in _conjuncts(child)]
    return [expr]


def _schema(con):
    """table -> {column: declared nullable?} for every table and view."""
    df = con.execute("SELECT lower(table_name) AS t, lower(column_name) AS c, "
                     "is_nullable AS nullable FROM duckdb_columns()").df()
    schema = {}
    for table, column, nullable in df.itertuples(index=False):
        schema.setdefault(table, {})[column] = bool(nullable)
    return schema


def _flatten_from(ref, schema, tables, joins, notes, con=None):   # CHANGED: takes con (see below)
    """Walk the FROM tree; fill tables {alias: info} and joins; return aliases below ref."""
    kind = ref.get("type")
    if kind == "BASE_TABLE":
        name = ref["table_name"]
        alias = (ref.get("alias") or name).lower()
        if name.lower() not in schema:
            notes.append(f"'{name}' is not a table or view (CTE?), so it was not analysed")
            return {alias}
        parts = [p for p in (ref.get("catalog_name"), ref.get("schema_name")) if p] + [name]
        tables[alias] = {"table": name.lower(), "cols": schema[name.lower()],
                         "sql": ".".join(_q(p) for p in parts)}
        return {alias}
    if kind == "JOIN":
        left = _flatten_from(ref["left"], schema, tables, joins, notes, con)     # CHANGED: pass con
        right = _flatten_from(ref["right"], schema, tables, joins, notes, con)   # CHANGED: pass con
        joins.append({"ref": ref, "left": left, "right": right})
        return left | right
    # CHANGED: a subquery in FROM is profiled as a derived table instead of being skipped.
    # The fix for the $1,240 (roll orders up to one row per customer, then join) has exactly
    # this shape, and without this the reviewer can only call that fix "not analysed".
    # Its columns count as nullable, because a derived table carries no constraints.
    if kind == "SUBQUERY" and con is not None:
        alias = (ref.get("alias") or "").lower()
        try:
            shell = _parse(con, "SELECT 1")
            shell["statements"][0]["node"] = ref["subquery"]["node"]
            body = con.execute("SELECT json_deserialize_sql(?::JSON)",
                               [json.dumps(shell)]).fetchone()[0].strip().rstrip(";")
            cols = [d[0].lower() for d in con.execute(f"SELECT * FROM ({body}) LIMIT 0").description]
        except Exception:
            cols = None   # fall through to the original "not analysed" note
        if alias and cols is not None:
            tables[alias] = {"table": f"subquery {alias}", "cols": dict.fromkeys(cols, True),
                             "sql": f"({body})"}
            return {alias}
    if kind in (None, "EMPTY", "EMPTY_FROM"):
        return set()
    notes.append(f"FROM item of type {kind} (subquery, table function, ...) was not analysed")
    return {(ref.get("alias") or str(kind)).lower()}


def _leftmost_alias(ref):
    while ref.get("type") == "JOIN":
        ref = ref["left"]
    if ref.get("type") == "BASE_TABLE":
        return (ref.get("alias") or ref["table_name"]).lower()
    return None


def _resolve(colref, tables):
    """COLUMN_REF -> (alias, column), or None if unknown or ambiguous."""
    names = [n.lower() for n in colref.get("column_names", [])]
    if not names:
        return None
    col = names[-1]
    if len(names) >= 2:
        qual = names[-2]
        hits = [qual] if qual in tables else [a for a, t in tables.items() if t["table"] == qual]
    else:
        hits = [a for a, t in tables.items() if col in t["cols"]]
    return (hits[0], col) if len(hits) == 1 else None


# ---------- check 1: filters on nullable columns ----------

def _null_contexts(tree, aware=False, soft=False):
    """Yield (column_ref, null_aware, soft) for every column in a WHERE tree.
    null_aware: inside IS [NOT] NULL, COALESCE/IFNULL, IS [NOT] DISTINCT FROM.
    soft:       under OR / CASE, where a NULL might not decide the row."""
    if isinstance(tree, list):
        for item in tree:
            yield from _null_contexts(item, aware, soft)
        return
    if not isinstance(tree, dict):
        return
    if "class" in tree:
        if tree["class"] == "SUBQUERY":
            return
        if tree["class"] == "COLUMN_REF":
            yield tree, aware, soft
            return
        if (tree.get("type") in _NULL_AWARE_TYPES
                or str(tree.get("function_name", "")).lower() in _NULL_AWARE_FUNCS):
            aware = True
        if tree.get("type") == "CONJUNCTION_OR" or tree["class"] == "CASE":
            soft = True
    for value in tree.values():
        if isinstance(value, (dict, list)):
            yield from _null_contexts(value, aware, soft)


# WHY A FILTER ON A NULLABLE COLUMN IS A CORRECTNESS ISSUE, NOT A STYLE ISSUE
#
# SQL predicates have three outcomes: TRUE, FALSE, UNKNOWN. Comparing anything
# with NULL gives UNKNOWN, so for a customer whose region is NULL:
#     region =  'West'        -> UNKNOWN
#     region <> 'West'        -> UNKNOWN
#     NOT (region = 'West')   -> UNKNOWN    (NOT UNKNOWN is still UNKNOWN)
# WHERE keeps only TRUE rows, so that customer lands in neither "West" nor
# "not West". There's no error and no row-count hint, just a normal-looking number.
#
# A style issue is one where a rewrite changes how the query reads but not what
# it returns. This one changes what the number means: "average revenue of West
# customers" quietly becomes "average revenue of customers whose region was
# recorded as West". Segments stop adding up to the total. When the NULLs
# aren't random (yours are customers from before region was mandatory, an
# older cohort), the result can be biased, not just based on fewer rows. The
# same rule makes `x NOT IN (subquery)` return nothing if the subquery yields
# a single NULL.
#
# The reviewer can't know whether excluding unknowns was intended, so it only
# asks for the intent to be written down: IS [NOT] NULL, COALESCE/IFNULL or
# IS [NOT] DISTINCT FROM on the column counts as handled. Severity comes from
# the data: high if NULLs exist now, low if the column is only nullable.
def _check_null_filters(con, where, tables):
    found = [(_resolve(ref, tables), aware, soft) for ref, aware, soft in _null_contexts(where)]
    handled = {key for key, aware, _ in found if key and aware}
    flagged = {}   # (alias, column) -> True if the NULL rows are certainly dropped
    for key, aware, soft in found:
        if key and not aware and key not in handled:
            flagged[key] = flagged.get(key, False) or not soft

    out = []
    for (alias, col), certain in flagged.items():
        info = tables[alias]
        if not info["cols"].get(col, True):
            continue   # declared NOT NULL / primary key: nothing unknown to drop
        n_null, n_total = con.execute(
            f"SELECT COUNT(*) - COUNT({_q(col)}), COUNT(*) FROM {info['sql']}").fetchone()
        share = n_null / n_total if n_total else 0.0
        target = f"{info['table']}.{col}"
        if n_null == 0:
            severity = "low"
            message = (f"WHERE filters on nullable {target}, which has no NULLs today; "
                       f"the first one that arrives will be dropped without an error.")
        else:
            severity = "high" if certain else "medium"
            message = (f"WHERE filters on nullable {target}: {n_null:,} of {n_total:,} rows "
                       f"({share:.1%}) are NULL, evaluate to UNKNOWN, and are "
                       f"{'silently excluded' if certain else 'possibly excluded'}.")
        out.append(QueryWarning(NULLABLE_FILTER_DROPS_UNKNOWN, severity, message, {
            "table": info["table"], "column": col, "alias": alias,
            "null_rows": int(n_null), "total_rows": int(n_total),
            "null_share": round(share, 4), "drop_is_certain": certain,
            "fix": (f"state the intent: '{alias}.{col} IS NOT NULL AND ...' if unknowns are "
                    f"meant to be excluded; COALESCE({alias}.{col}, <default>) or "
                    f"'... OR {alias}.{col} IS NULL' if they are not."),
        }))
    return out


# ---------- check 2: joins that change row count ----------

def _join_keys(ref, left, right, tables):
    """[(left_alias, left_col, right_alias, right_col), ...] for a pure equi-join, else None."""
    pairs = []
    if ref.get("using_columns"):
        for col in (c.lower() for c in ref["using_columns"]):
            l = [a for a in left if a in tables and col in tables[a]["cols"]]
            r = [a for a in right if a in tables and col in tables[a]["cols"]]
            if len(l) != 1 or len(r) != 1:
                return None
            pairs.append((l[0], col, r[0], col))
    elif ref.get("condition"):
        for cond in _conjuncts(ref["condition"]):
            if cond.get("type") != "COMPARE_EQUAL":
                return None
            a = _resolve(cond["left"], tables) if cond["left"].get("class") == "COLUMN_REF" else None
            b = _resolve(cond["right"], tables) if cond["right"].get("class") == "COLUMN_REF" else None
            if a is None or b is None:
                return None
            if a[0] in left and b[0] in right:
                pairs.append(a + b)
            elif b[0] in left and a[0] in right:
                pairs.append(b + a)
            else:
                return None
    else:
        return None   # cross / comma / natural join
    # The histogram trick needs each side's keys to come from one table.
    if len({p[0] for p in pairs}) != 1 or len({p[2] for p in pairs}) != 1:
        return None
    return pairs


# HOW FAN-OUT IS DETECTED WITHOUT RUNNING THE JOIN (LET ALONE TWICE)
#
# The obvious check runs the query, then runs it again without the join (or
# with COUNT(DISTINCT customer_id) beside COUNT(*)) and compares. That runs
# the expensive part twice, and at scale the fanned-out join *is* the
# expensive part.
#
# For an equi-join, the output size depends only on how many rows each side
# has per key value:
#
#     rows(L JOIN R ON L.k = R.k) = SUM over k of n_L(k) * n_R(k)
#
# So each side is collapsed to a histogram (one row per key, with its count),
# the two histograms are joined (one row per distinct key: about 5,000 here,
# not one per order), and everything is read off that small result:
#     inner-join rows   SUM(n_L * n_R)          exact, not an optimizer estimate
#     fan-out           MAX(n_R) > 1 on a matched key -> some left rows repeat
#     rows repeated     SUM(n_L) where n_R > 1
#     orphans           left rows whose key never appears on the right
#                       (includes NULL keys: NULL = NULL is UNKNOWN, never a match)
# Single-table WHERE conjuncts are pushed into the histograms with
# COUNT(*) FILTER (WHERE ...). One scan per table then yields both the filtered
# count (what your query sees, so rows_before/rows_after match its n_rows)
# and the unfiltered count (which separates "dropped by your filter" from
# "dropped because the key has no partner").
# Cost: one GROUP BY per table and a key-level join. The row-level join never runs.
def _profile_join(con, pairs, tables, filters):
    def histogram(alias, cols):
        keys = ", ".join(f"{_q(alias)}.{_q(c)} AS k{i}" for i, c in enumerate(cols))
        cond = " AND ".join(filters.get(alias, [])) or "TRUE"
        return (f"SELECT {keys}, COUNT(*) AS n_all, COUNT(*) FILTER (WHERE {cond}) AS n "
                f"FROM {tables[alias]['sql']} AS {_q(alias)} GROUP BY ALL")

    idx = range(len(pairs))
    on = " AND ".join(f"l.k{i} = r.k{i}" for i in idx)
    null_key = " OR ".join(f"k{i} IS NULL" for i in idx)
    sql = f"""
        WITH l AS ({histogram(pairs[0][0], [p[1] for p in pairs])}),
             r AS ({histogram(pairs[0][2], [p[3] for p in pairs])}),
             m AS (SELECT l.n AS ln, r.n AS rn FROM l JOIN r ON {on})
        SELECT * FROM
          (SELECT SUM(n) AS left_rows,
                  SUM(n) FILTER (WHERE {null_key}) AS left_null_key_rows FROM l) AS ls,
          (SELECT SUM(n) AS right_rows,
                  SUM(n) FILTER (WHERE {null_key}) AS right_null_key_rows FROM r) AS rs,
          (SELECT SUM(ln * rn)                  AS inner_join_rows,
                  SUM(ln)                       AS left_rows_key_exists,
                  SUM(rn)                       AS right_rows_key_exists,
                  SUM(ln) FILTER (WHERE rn > 0) AS left_rows_matched,
                  SUM(rn) FILTER (WHERE ln > 0) AS right_rows_matched,
                  SUM(ln) FILTER (WHERE rn > 1) AS left_rows_repeated,
                  SUM(rn) FILTER (WHERE ln > 1) AS right_rows_repeated,
                  MAX(rn) FILTER (WHERE ln > 0) AS max_matches_per_left_row,
                  MAX(ln) FILTER (WHERE rn > 0) AS max_matches_per_right_row
           FROM m) AS ms
    """
    row = con.execute(sql).df().fillna(0).iloc[0]
    return {name: int(value) for name, value in row.items()}


def _check_joins(con, where, tables, joins):
    """Profile each equi-join. Returns (warnings, fanned), where fanned maps
    alias -> joins that repeat that alias's rows."""
    # WHERE can't be pushed into the NULL-filling side of an outer join.
    null_side = set()
    for j in joins:
        jt = j["ref"].get("join_type")
        if jt == "LEFT":
            null_side |= j["right"]
        elif jt == "RIGHT":
            null_side |= j["left"]
        elif jt in ("OUTER", "FULL"):
            null_side |= j["left"] | j["right"]

    filters = {}
    for conj in _conjuncts(where):
        keys = [_resolve(c, tables) for c in _columns(conj)]
        aliases = {k[0] for k in keys if k}
        if (not keys or None in keys or len(aliases) != 1 or aliases & null_side
                or any(e["class"] == "SUBQUERY" for e in _exprs(conj))):
            continue
        try:
            filters.setdefault(aliases.pop(), []).append(_render(con, conj))
        except Exception:
            pass   # can't deparse: that side is profiled unfiltered

    out, fanned = [], {}
    for j in joins:
        ref = j["ref"]
        jt = ref.get("join_type") or "INNER"
        pairs = _join_keys(ref, j["left"], j["right"], tables)
        if pairs is None:
            out.append(QueryWarning(REVIEW_INCOMPLETE, "info",
                f"{jt} join without a plain column = column condition (cross, comma, natural "
                f"or non-equi join) was not profiled; its effect on row count is unknown.",
                {"join_type": jt}))
            continue
        label = " AND ".join(f"{a}.{x} = {b}.{y}" for a, x, b, y in pairs)
        try:
            s = _profile_join(con, pairs, tables, filters)
        except Exception as e:
            out.append(QueryWarning(REVIEW_INCOMPLETE, "info",
                                    f"could not profile join {label}: {e}"))
            continue

        left_unmatched = s["left_rows"] - s["left_rows_matched"]
        right_unmatched = s["right_rows"] - s["right_rows_matched"]
        rows_after = {
            "INNER": s["inner_join_rows"],
            "LEFT":  s["inner_join_rows"] + left_unmatched,
            "RIGHT": s["inner_join_rows"] + right_unmatched,
            "OUTER": s["inner_join_rows"] + left_unmatched + right_unmatched,
            "FULL":  s["inner_join_rows"] + left_unmatched + right_unmatched,
            "SEMI":  s["left_rows_matched"],
            "ANTI":  left_unmatched,
        }.get(jt, s["inner_join_rows"])
        multiplies = jt not in ("SEMI", "ANTI")

        # Record whose rows get repeated, and by how much (used by check 3).
        if multiplies:
            for side in ("left", "right"):
                most = s[f"max_matches_per_{side}_row"]
                if most > 1:
                    for alias in j[side]:
                        fanned.setdefault(alias, []).append({
                            "join": label, "max_copies": most,
                            "avg_copies": round(s["inner_join_rows"] / s[f"{side}_rows_matched"], 2)})

        repeated = s["left_rows_repeated"] if multiplies else 0
        orphans = (s["left_rows"] - s["left_rows_key_exists"]) if jt in ("INNER", "RIGHT", "SEMI") else 0
        if repeated or orphans:
            lt, rt = tables[pairs[0][0]]["table"], tables[pairs[0][2]]["table"]
            reasons = []
            if repeated:
                reasons.append(f"{repeated:,} {lt} rows match more than one {rt} row "
                               f"(up to {s['max_matches_per_left_row']:,})")
            if orphans:
                reasons.append(f"{orphans:,} {lt} rows have no {rt} row at all "
                               f"({s['left_null_key_rows']:,} with a NULL key) and vanish")
            out.append(QueryWarning(JOIN_ROW_COUNT_CHANGE, "medium",
                f"{jt} JOIN on {label} turns {s['left_rows']:,} {lt} rows into "
                f"{rows_after:,}: " + "; ".join(reasons) + ".",
                {"join": label, "join_type": jt, "left_table": lt, "right_table": rt,
                 "rows_before": s["left_rows"], "rows_after": rows_after,
                 "ratio": round(rows_after / s["left_rows"], 3) if s["left_rows"] else None,
                 "filters_applied": {a: filters.get(a, []) for a in (pairs[0][0], pairs[0][2])},
                 **s}))
    return out, fanned


# ---------- check 3: aggregates over a fanned-out result ----------

def _check_aggregates(con, node, tables, fanned, driving):
    agg_names = {r[0].lower() for r in con.execute(
        "SELECT DISTINCT function_name FROM duckdb_functions() "
        "WHERE function_type = 'aggregate'").fetchall()} | {"count_star"}
    out = []
    for clause in ("select_list", "having"):
        for agg in _exprs(node.get(clause)):
            name = str(agg.get("function_name", "")).lower()
            if agg["class"] != "FUNCTION" or name not in agg_names:
                continue
            if agg.get("distinct") or name in _DUP_INSENSITIVE:
                continue   # MIN, MAX, COUNT(DISTINCT ...) are unchanged by repeated rows
            keys = [_resolve(c, tables) for c in _columns(agg.get("children"))]
            # COUNT(*) / SUM(1) read no column: they count rows of the driving table.
            aliases = {k[0] for k in keys if k} or ({driving} if driving else set())
            # SUM(o.qty * c.price) is fine while o is at its own grain, so flag
            # only when every table the aggregate reads from has been fanned out.
            if not aliases or not all(a in fanned for a in aliases):
                continue
            joins = [e for a in sorted(aliases) for e in fanned[a]]
            worst = max(joins, key=lambda e: e["max_copies"])
            fanned_tables = sorted({tables[a]["table"] for a in aliases if a in tables})
            if name == "count_star" or not agg.get("children"):
                shown = "COUNT(*)"
            else:
                try:
                    shown = _render(con, agg)
                except Exception:
                    shown = name
            if name in ("avg", "mean"):
                effect = "is weighted by how many times each row was repeated"
            elif name in ("sum", "count", "count_star"):
                effect = "is inflated"
            else:
                effect = "is computed over repeated rows"
            out.append(QueryWarning(AGGREGATE_OVER_FANOUT, "high",
                f"{shown} {effect}: {', '.join(fanned_tables)} rows are repeated by the join "
                f"on {worst['join']} (avg {worst['avg_copies']}x, max {worst['max_copies']:,}x).",
                {"aggregate": name, "expression": shown,
                 "output_column": agg.get("alias") or None,
                 "fanned_tables": fanned_tables, "joins": joins,
                 "fix": ("compute it at the intended grain: pre-aggregate the other table to one "
                         "row per join key, or use EXISTS if the join is only a filter.")}))
    return out


# ---------- entry point ----------

def review_query(con: duckdb.DuckDBPyConnection, sql: str) -> list[QueryWarning]:
    """Review one SELECT against the live data in `con`. Returns QueryWarning objects."""
    try:
        parsed = _parse(con, sql)
    except Exception as e:
        return [QueryWarning(REVIEW_INCOMPLETE, "info", f"could not parse query: {e}")]
    statements = parsed.get("statements") or []
    if parsed.get("error") or len(statements) != 1:
        return [QueryWarning(REVIEW_INCOMPLETE, "info",
                             parsed.get("error_message") or "expected exactly one SELECT")]
    node = statements[0]["node"]
    if node.get("type") != "SELECT_NODE":
        return [QueryWarning(REVIEW_INCOMPLETE, "info",
                             f"{node.get('type')} (e.g. UNION) is not analysed")]

    tables, joins, notes = {}, [], []
    from_table = node.get("from_table") or {}
    _flatten_from(from_table, _schema(con), tables, joins, notes, con)   # CHANGED: pass con
    if (node.get("cte_map") or {}).get("map"):
        notes.append("WITH clause present; CTE bodies are not analysed")
    warnings = [QueryWarning(REVIEW_INCOMPLETE, "info", n) for n in notes]

    where = node.get("where_clause")
    warnings += _check_null_filters(con, where, tables)
    join_warnings, fanned = _check_joins(con, where, tables, joins)
    warnings += join_warnings
    warnings += _check_aggregates(con, node, tables, fanned, _leftmost_alias(from_table))
    return warnings

In [20]:
# Run review_query on your Phase 1 and Phase 2 queries.
# con_board is the warehouse as it stood when the $1,240 went to the board: the same two tables,
# before Step 2.1 put NULLs into region. con is the warehouse as it is now.
con_board = load_warehouse(duckdb.connect(), customers_df, orders_df)

# Three probes beyond the brief's queries, each aimed at one thing a reviewer should get right.
PROBE_CTE = """
WITH west_rows AS (
    SELECT c.customer_id, c.revenue
    FROM customers AS c
    JOIN orders AS o ON o.customer_id = c.customer_id
    WHERE c.region = 'West'
)
SELECT AVG(revenue) AS avg_revenue
FROM west_rows
""".strip()   # the $1,240 bug again, moved inside a CTE

PROBE_GROUPED = """
SELECT c.customer_id, AVG(c.revenue) AS revenue, COUNT(*) AS n_orders
FROM customers AS c
JOIN orders AS o ON o.customer_id = c.customer_id
GROUP BY c.customer_id
""".strip()   # the join fans out, but GROUP BY returns to one row per customer, so AVG is right

PROBE_DISTINCT = """
SELECT COUNT(DISTINCT c.customer_id) AS customers_with_orders
FROM customers AS c
JOIN orders AS o ON o.customer_id = c.customer_id
""".strip()   # the join fans out, but COUNT(DISTINCT) ignores repeated rows

SHORT = {JOIN_ROW_COUNT_CHANGE: "join", NULLABLE_FILTER_DROPS_UNKNOWN: "null",
         AGGREGATE_OVER_FANOUT: "aggregate", REVIEW_INCOMPLETE: "incomplete"}
CASES = [   # (query, warehouse, sql, what a careful human reviewer should flag)
    ("P1 Finance, board-time warehouse",      con_board, Q_FINANCE,   {"null"}),
    ("P1 Dashboard, board-time warehouse",    con_board, Q_DASHBOARD, {"null", "join", "aggregate"}),
    ("P1 Finance",                            con, Q_FINANCE,         {"null"}),
    ("P1 Dashboard",                          con, Q_DASHBOARD,       {"null", "join", "aggregate"}),
    ("P1 orders rolled up, then joined",      con, Q_FIXED,           {"null"}),
    ("P2 not-West, naive (!=)",               con, Q_NOT_WEST_NAIVE,  {"null"}),
    ("P2 not-West, meant (IS DISTINCT FROM)", con, Q_NOT_WEST_MEANT,  set()),
    ("P2 window, region average on each row", con, Q_WINDOW,          set()),
    ("probe: fan-out inside a CTE",           con, PROBE_CTE,         {"null", "join", "aggregate"}),
    ("probe: fan-out, regrouped per customer", con, PROBE_GROUPED,    {"join"}),
    ("probe: fan-out, COUNT(DISTINCT)",       con, PROBE_DISTINCT,    {"join"}),
]

results, rows = {}, []
for label, warehouse, sql, expected in CASES:
    found = review_query(warehouse, sql)
    results[label] = found
    got = {SHORT[w.code] for w in found} - {"incomplete"}
    rows.append({
        "query": label,
        "raised": ", ".join(f"{SHORT[w.code]} ({w.severity})" for w in found) or "-",
        "caught": ", ".join(sorted(expected & got)) or "-",
        "missed": ", ".join(sorted(expected - got)) or "-",
        "false alarm": ", ".join(sorted(got - expected)) or "-",
        "blocked by a no-high-warnings gate": any(w.severity == "high" for w in found),
    })
scorecard = pd.DataFrame(rows)
display(scorecard)

# The warnings in full for the query that reached the board, on the warehouse it ran against.
board = results["P1 Dashboard, board-time warehouse"]
display(warnings_frame(board))
join_w = next((w for w in board if w.code == JOIN_ROW_COUNT_CHANGE), None)
if join_w is not None:
    print(f"reviewer's rows before -> after the join: {join_w.details['rows_before']:,} -> "
          f"{join_w.details['rows_after']:,} | Phase 1 n_rows: {finance_rows:,} -> {dashboard_rows:,}")

# And the miss: what comes back for the same bug written as a CTE.
display(warnings_frame(results["probe: fan-out inside a CTE"]))

,query,raised,caught,missed,false alarm,blocked by a no-high-warnings gate
0,"P1 Finance, board-time warehouse",null (low),null,-,-,False
1,"P1 Dashboard, board-time warehouse","null (low), join (medium), aggregate (high), aggregate (high)","aggregate, join, null",-,-,True
2,P1 Finance,null (high),null,-,-,True
3,P1 Dashboard,"null (high), join (medium), aggregate (high), aggregate (high)","aggregate, join, null",-,-,True
4,"P1 orders rolled up, then joined",null (high),null,-,-,True
5,"P2 not-West, naive (!=)",null (high),null,-,-,True
6,"P2 not-West, meant (IS DISTINCT FROM)",-,-,-,-,False
7,"P2 window, region average on each row",-,-,-,-,False
8,probe: fan-out inside a CTE,"incomplete (info), incomplete (info)",-,"aggregate, join, null",-,False
9,"probe: fan-out, regrouped per customer","join (medium), aggregate (high), aggregate (high)",join,-,aggregate,True


,code,severity,message,details
0,NULLABLE_FILTER_DROPS_UNKNOWN,low,"WHERE filters on nullable customers.region, which has no NULLs today; the first one that arrives will be dropped without an error.","{'table': 'customers', 'column': 'region', 'alias': 'c', 'null_rows': 0, 'total_rows': 5000, 'null_share': 0.0, 'drop_is_certain': True,..."
1,JOIN_ROW_COUNT_CHANGE,medium,"INNER JOIN on c.customer_id = o.customer_id turns 3,000 customers rows into 19,618: 2,422 customers rows match more than one orders row ...","{'join': 'c.customer_id = o.customer_id', 'join_type': 'INNER', 'left_table': 'customers', 'right_table': 'orders', 'rows_before': 3000,..."
2,AGGREGATE_OVER_FANOUT,high,avg(c.revenue) is weighted by how many times each row was repeated: customers rows are repeated by the join on c.customer_id = o.custome...,"{'aggregate': 'avg', 'expression': 'avg(c.revenue)', 'output_column': 'avg_revenue', 'fanned_tables': ['customers'], 'joins': [{'join': ..."
3,AGGREGATE_OVER_FANOUT,high,"COUNT(*) is inflated: customers rows are repeated by the join on c.customer_id = o.customer_id (avg 6.54x, max 54x).","{'aggregate': 'count_star', 'expression': 'COUNT(*)', 'output_column': 'n_rows', 'fanned_tables': ['customers'], 'joins': [{'join': 'c.c..."


reviewer's rows before -> after the join: 3,000 -> 19,618 | Phase 1 n_rows: 3,000 -> 19,618


,code,severity,message,details
0,REVIEW_INCOMPLETE,info,"'west_rows' is not a table or view (CTE?), so it was not analysed",{}
1,REVIEW_INCOMPLETE,info,WITH clause present; CTE bodies are not analysed,{}


In [21]:
# ---- The model's own tests (Cell 2 of its reply), pasted unchanged. They encode its predictions ----
# ---- for my setup, so a failing assert here is a wrong prediction, and I would report it as one. ----
FINANCE_SQL = """
SELECT AVG(c.revenue) AS avg_revenue,
       COUNT(*)       AS n_rows
FROM customers AS c
WHERE c.region = 'West'
"""

DASHBOARD_SQL = """
SELECT AVG(c.revenue) AS avg_revenue,
       COUNT(*)       AS n_rows
FROM customers AS c
JOIN orders AS o
  ON o.customer_id = c.customer_id
WHERE c.region = 'West'
"""

fin = review_query(con, FINANCE_SQL)
dash = review_query(con, DASHBOARD_SQL)
display(warnings_frame(fin))
display(warnings_frame(dash))

def codes(ws):
    return sorted(w.code for w in ws)

# 1. Finance: only the NULL-region warning.
assert codes(fin) == [NULLABLE_FILTER_DROPS_UNKNOWN], codes(fin)

# 2. Dashboard: NULL warning, row-count change, and fan-out on both AVG and COUNT(*).
assert codes(dash) == sorted([NULLABLE_FILTER_DROPS_UNKNOWN, JOIN_ROW_COUNT_CHANGE,
                              AGGREGATE_OVER_FANOUT, AGGREGATE_OVER_FANOUT]), codes(dash)

# 3. The histogram counts must reconcile exactly with each query's own n_rows.
join_w = next(w for w in dash if w.code == JOIN_ROW_COUNT_CHANGE)
assert join_w.details["rows_before"] == con.execute(FINANCE_SQL).fetchone()[1]
assert join_w.details["rows_after"] == con.execute(DASHBOARD_SQL).fetchone()[1]

# 4. Same bug written from the orders side: the orders row count is preserved,
#    but customer rows still repeat, so the aggregate check must still fire.
FLIPPED_SQL = """
SELECT AVG(c.revenue) FROM orders AS o
JOIN customers AS c ON c.customer_id = o.customer_id
WHERE c.region = 'West'
"""
assert AGGREGATE_OVER_FANOUT in codes(review_query(con, FLIPPED_SQL))

# 5. The fixes clear their warnings.
EXISTS_SQL = """
SELECT AVG(c.revenue) AS avg_revenue, COUNT(*) AS n_rows
FROM customers AS c
WHERE c.region = 'West'
  AND EXISTS (SELECT 1 FROM orders AS o WHERE o.customer_id = c.customer_id)
"""
assert codes(review_query(con, EXISTS_SQL)) == [NULLABLE_FILTER_DROPS_UNKNOWN]

EXPLICIT_SQL = """
SELECT AVG(c.revenue) AS avg_revenue, COUNT(*) AS n_rows
FROM customers AS c
WHERE c.region IS NOT NULL AND c.region = 'West'
"""
assert review_query(con, EXPLICIT_SQL) == []
print("all checks passed")

,code,severity,message,details
0,NULLABLE_FILTER_DROPS_UNKNOWN,high,"WHERE filters on nullable customers.region: 731 of 5,000 rows (14.6%) are NULL, evaluate to UNKNOWN, and are silently excluded.","{'table': 'customers', 'column': 'region', 'alias': 'c', 'null_rows': 731, 'total_rows': 5000, 'null_share': 0.1462, 'drop_is_certain': ..."


,code,severity,message,details
0,NULLABLE_FILTER_DROPS_UNKNOWN,high,"WHERE filters on nullable customers.region: 731 of 5,000 rows (14.6%) are NULL, evaluate to UNKNOWN, and are silently excluded.","{'table': 'customers', 'column': 'region', 'alias': 'c', 'null_rows': 731, 'total_rows': 5000, 'null_share': 0.1462, 'drop_is_certain': ..."
1,JOIN_ROW_COUNT_CHANGE,medium,"INNER JOIN on c.customer_id = o.customer_id turns 2,559 customers rows into 16,568: 2,059 customers rows match more than one orders row ...","{'join': 'c.customer_id = o.customer_id', 'join_type': 'INNER', 'left_table': 'customers', 'right_table': 'orders', 'rows_before': 2559,..."
2,AGGREGATE_OVER_FANOUT,high,avg(c.revenue) is weighted by how many times each row was repeated: customers rows are repeated by the join on c.customer_id = o.custome...,"{'aggregate': 'avg', 'expression': 'avg(c.revenue)', 'output_column': 'avg_revenue', 'fanned_tables': ['customers'], 'joins': [{'join': ..."
3,AGGREGATE_OVER_FANOUT,high,"COUNT(*) is inflated: customers rows are repeated by the join on c.customer_id = o.customer_id (avg 6.47x, max 54x).","{'aggregate': 'count_star', 'expression': 'COUNT(*)', 'output_column': 'n_rows', 'fanned_tables': ['customers'], 'joins': [{'join': 'c.c..."


all checks passed


**Would it have caught the \$1,240 before it reached the board? Yes.** On the board-time warehouse, the dashboard query raises `AGGREGATE_OVER_FANOUT` at high severity twice, once for `AVG(c.revenue)` and once for `COUNT(*)`, plus `JOIN_ROW_COUNT_CHANGE`. The reviewer's row counts reconcile exactly with Phase 1's: 3,000 customers before the join, 19,618 rows after. The Finance query raises nothing about joins. So the one difference between the two queries is exactly what it flags, and it gets there without knowing that \$812 exists. It also never runs the row-level join. It joins one-row-per-key summaries of each side, which is the trick its comment explains. The model's own tests pass on my data (the last code cell prints *all checks passed*), so its hand-traced predictions held. It predicted "about 750" NULL rows; there are 731.

**What it caught.**
- **The null filter.** `WHERE region != 'West'` gets a high-severity `NULLABLE_FILTER_DROPS_UNKNOWN` that names the 731 NULL rows (14.6%), and the version written with `IS DISTINCT FROM` passes clean. On the board-time warehouse, before any NULLs existed, the same kind of filter gets a *low* warning that the first NULL to arrive will be dropped. That warning would have flagged the Phase 2 problem before it happened.
- **Correct silences.** The window query (no filter, no join) gets nothing. `COUNT(DISTINCT customer_id)` over the fanned join gets the join warning but no aggregate warning. That is right, because repeated rows do not change a distinct count. With my change (below), the rolled-up join `Q_FIXED` passes both fan-out checks and keeps only the null warning.

**What it missed or got wrong.**
- **A fan-out inside a CTE gets through.** The dashboard's join moved into a `WITH` clause gives the same \$1,240. The reviewer returns only `REVIEW_INCOMPLETE` notes at *info* severity, with no join, aggregate or null warning. To its credit it never calls the query clean. But a publish gate that blocks on high warnings, the gate the model itself recommends, lets this query through. Analysts write CTEs constantly, so this is the biggest gap.
- **A false alarm on a correct query.** When the fanned join is grouped back to one row per customer (`GROUP BY c.customer_id`), each `AVG(c.revenue)` is just that customer's revenue, and it is correct. `COUNT(*)` is that customer's order count, which is exactly what the query wants. The reviewer still raises two high `AGGREGATE_OVER_FANOUT` warnings, one for each. The model listed this limit itself.
- **It treats `=` and `!=` alike.** The Finance query's `region = 'West'` gets the same high null warning as `region != 'West'`. That is defensible, because the Finance number also leaves out the 441 West customers whose region is NULL. But they are different mistakes. `!=` silently changes the question, since "everyone but West" loses the unknowns. `=` can only ever mean "recorded as West". A gate that blocks on high warnings therefore blocks the correct \$812, and every other regional query in the company, until each one states `region IS NOT NULL`.
- **As returned, it could not confirm the fix.** The model's `_flatten_from` sets aside any FROM item that is not a base table. The correct way to bring orders in, rolling them up to one row per customer and then joining (Step 1.3's `Q_FIXED`), would come back only as `REVIEW_INCOMPLETE`, with a message blaming a non-equi join that is not there. Profiling the subquery as a derived table fixes it. This is my one change to the code, logged in the transcript.

### Task 4.2: The one-page writeup (8 pts)

**To:** the Director · **Re:** \$1,240 or \$812?

**Which number is right.** \$812. "Average revenue per customer" should count each West customer once, and Finance's query does.

**Why the two differed.** The dashboard's query attached every order to its customer before averaging, so a customer with 40 orders was counted 40 times. In our rebuild, 3,000 West customers became 19,618 rows. Our biggest spenders order most, so they dominated the average, pushing it up by half. Nobody's arithmetic was wrong. The query answered a different question.

**Does the programme work?** The regional gap cannot tell us. West was given the larger accounts, and larger accounts spend more with or without an account manager. In a simulation where we know the programme adds \$40 per customer, the raw West-versus-rest gap was \$386. Comparing accounts of similar size recovered \$38. Also controlling for support tickets, which the programme itself generates, flipped it to −\$30. Real data can only be adjusted for what we measured, so the honest answer is "not yet known". A pilot in another region, with accounts assigned to the programme at random, would settle it.

**Before a number goes on a slide:**
1. Every metric names its unit (customer, order), and an automated check confirms one row per unit; a join that adds rows blocks publication.
2. Regional reports show Unknown as its own line, and `!=` on a nullable column fails review.
3. Causal claims control only for things measured before the programme began.

*Word count: 244 (limit 250).*

### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

**Model and date:** Claude Opus 5.5 (Anthropic, model ID `claude-opus-5-5`), 2 November 2026.

**How the prompt was sent.** The prompt went to a separate, fresh Claude session started from Claude Code. That session had no access to this notebook, my files or any earlier conversation, so the reply answers the prompt exactly as written. A Claude Code session has to be told to act as an ordinary chat assistant, so the prompt was sent inside a short wrapper. The brief's P.R.I.M.E. prompt is unchanged. Below it I added a *Context* block with my table schemas and my two Phase 1 queries, because the prompt refers to "my table names" and "my Phase 1 queries" and a fresh session cannot see either. The full text as sent, word for word:

```text
You are being used as a plain chat assistant. Reply to the user message below exactly as you would in an ordinary chat conversation. Write your whole answer directly in your reply. Do not use any tools, and do not read, search or run anything on this machine. Your final reply is passed back verbatim as the chat answer.

=== USER MESSAGE ===
[Prep] Act as an analytics engineer who reviews SQL before it reaches
a dashboard.

[Request] Write a Python function review_query(con, sql) that runs
against DuckDB and returns a list of warnings for a given SELECT. It
should flag at minimum: a join that changes the row count, a filter on
a nullable column that will silently drop unknowns, and an aggregate
computed over a fanned-out result.

[Iterate] duckdb and pandas only. Reuse my table names, customers and
orders. Return structured warnings, not printed text, so I can test it.

[Mechanism Check] Add comments explaining how you detect fan-out
without running the full join twice, and why three-valued logic makes
the null filter a correctness issue rather than a style issue.

[Evaluate] Run it against both of my Phase 1 queries and tell me
whether it would have caught the $1,240 before it reached the board.

Context on my setup:
- Google Colab, an in-memory DuckDB connection called con.
- customers(customer_id INTEGER PRIMARY KEY, region VARCHAR, revenue DOUBLE, signup_date DATE): 5,000 rows, one per customer. region has no NOT NULL constraint, and in the current warehouse about 15% of rows have region NULL (customers onboarded before the field was mandatory).
- orders(order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date DATE): one row per order, about 34,000 rows. Every customer has at least one order; the mean is about 7 per customer with a long right tail.
- My two Phase 1 queries, average revenue per customer in the West region:

Finance query (gives about $812):
SELECT AVG(c.revenue) AS avg_revenue,
       COUNT(*)       AS n_rows
FROM customers AS c
WHERE c.region = 'West'

Dashboard query (gives about $1,240):
SELECT AVG(c.revenue) AS avg_revenue,
       COUNT(*)       AS n_rows
FROM customers AS c
JOIN orders AS o
  ON o.customer_id = c.customer_id
WHERE c.region = 'West'
=== END USER MESSAGE ===
```

**The reply, exactly as it came back.** It is shown as a quote. The only difference from the raw text is that dollar signs outside code are written `\$`, so that Colab does not read them as maths.

> I can't run anything against your Colab session from here, so the Evaluate section traces by hand what this code returns for the setup you described. Cell 2's asserts encode those predictions. If one fails, paste the traceback.
>
> ## Cell 1: the reviewer
>
> It uses DuckDB's own SQL parser (`json_serialize_sql`) rather than regex, so it needs duckdb 0.10 or later. Besides duckdb and pandas it imports only the standard library (`json`, `re`, `dataclasses`).
>
> ```python
> import json
> import re
> from dataclasses import dataclass, field, asdict
>
> import duckdb
> import pandas as pd
>
> JOIN_ROW_COUNT_CHANGE = "JOIN_ROW_COUNT_CHANGE"
> NULLABLE_FILTER_DROPS_UNKNOWN = "NULLABLE_FILTER_DROPS_UNKNOWN"
> AGGREGATE_OVER_FANOUT = "AGGREGATE_OVER_FANOUT"
> REVIEW_INCOMPLETE = "REVIEW_INCOMPLETE"   # something wasn't analysed; never a silent pass
>
>
> @dataclass
> class QueryWarning:
>     code: str        # one of the constants above; assert on this in tests
>     severity: str    # "high" | "medium" | "low" | "info"
>     message: str     # one-line explanation
>     details: dict = field(default_factory=dict)   # the numbers behind the message
>
>
> def warnings_frame(warnings):
>     """One row per warning, for display() in Colab."""
>     return pd.DataFrame([asdict(w) for w in warnings],
>                         columns=["code", "severity", "message", "details"])
>
>
> # Aggregates whose answer doesn't change when input rows are repeated.
> _DUP_INSENSITIVE = {"min", "max", "any_value", "arbitrary", "first", "last",
>                     "arg_min", "arg_max", "argmin", "argmax", "min_by", "max_by",
>                     "bool_and", "bool_or", "bit_and", "bit_or", "approx_count_distinct"}
> # Constructs showing the author decided what happens to NULLs.
> _NULL_AWARE_TYPES = {"OPERATOR_IS_NULL", "OPERATOR_IS_NOT_NULL", "OPERATOR_COALESCE",
>                      "COMPARE_DISTINCT_FROM", "COMPARE_NOT_DISTINCT_FROM"}
> _NULL_AWARE_FUNCS = {"coalesce", "ifnull", "nvl"}
>
>
> # ---------- parsing helpers (DuckDB's parser and deparser) ----------
>
> def _parse(con, sql):
>     return json.loads(con.execute("SELECT json_serialize_sql(?)", [sql]).fetchone()[0])
>
>
> def _render(con, expr):
>     """Expression AST -> SQL text, via DuckDB's deparser."""
>     shell = _parse(con, "SELECT 1")
>     shell["statements"][0]["node"]["select_list"] = [dict(expr, alias="")]
>     text = con.execute("SELECT json_deserialize_sql(?::JSON)",
>                        [json.dumps(shell)]).fetchone()[0]
>     return re.sub(r"^\s*SELECT\s+", "", text.strip().rstrip(";"), flags=re.IGNORECASE)
>
>
> def _q(name):
>     return '"' + str(name).replace('"', '""') + '"'
>
>
> def _exprs(tree):
>     """Every expression node under tree; does not enter subqueries (different scope)."""
>     if isinstance(tree, list):
>         for item in tree:
>             yield from _exprs(item)
>     elif isinstance(tree, dict):
>         if "class" in tree:
>             yield tree
>             if tree["class"] == "SUBQUERY":
>                 return
>         for value in tree.values():
>             if isinstance(value, (dict, list)):
>                 yield from _exprs(value)
>
>
> def _columns(tree):
>     return [e for e in _exprs(tree) if e["class"] == "COLUMN_REF"]
>
>
> def _conjuncts(expr):
>     if not expr:
>         return []
>     if expr.get("type") == "CONJUNCTION_AND":
>         return [c for child in expr.get("children", []) for c in _conjuncts(child)]
>     return [expr]
>
>
> def _schema(con):
>     """table -> {column: declared nullable?} for every table and view."""
>     df = con.execute("SELECT lower(table_name) AS t, lower(column_name) AS c, "
>                      "is_nullable AS nullable FROM duckdb_columns()").df()
>     schema = {}
>     for table, column, nullable in df.itertuples(index=False):
>         schema.setdefault(table, {})[column] = bool(nullable)
>     return schema
>
>
> def _flatten_from(ref, schema, tables, joins, notes):
>     """Walk the FROM tree; fill tables {alias: info} and joins; return aliases below ref."""
>     kind = ref.get("type")
>     if kind == "BASE_TABLE":
>         name = ref["table_name"]
>         alias = (ref.get("alias") or name).lower()
>         if name.lower() not in schema:
>             notes.append(f"'{name}' is not a table or view (CTE?), so it was not analysed")
>             return {alias}
>         parts = [p for p in (ref.get("catalog_name"), ref.get("schema_name")) if p] + [name]
>         tables[alias] = {"table": name.lower(), "cols": schema[name.lower()],
>                          "sql": ".".join(_q(p) for p in parts)}
>         return {alias}
>     if kind == "JOIN":
>         left = _flatten_from(ref["left"], schema, tables, joins, notes)
>         right = _flatten_from(ref["right"], schema, tables, joins, notes)
>         joins.append({"ref": ref, "left": left, "right": right})
>         return left | right
>     if kind in (None, "EMPTY", "EMPTY_FROM"):
>         return set()
>     notes.append(f"FROM item of type {kind} (subquery, table function, ...) was not analysed")
>     return {(ref.get("alias") or str(kind)).lower()}
>
>
> def _leftmost_alias(ref):
>     while ref.get("type") == "JOIN":
>         ref = ref["left"]
>     if ref.get("type") == "BASE_TABLE":
>         return (ref.get("alias") or ref["table_name"]).lower()
>     return None
>
>
> def _resolve(colref, tables):
>     """COLUMN_REF -> (alias, column), or None if unknown or ambiguous."""
>     names = [n.lower() for n in colref.get("column_names", [])]
>     if not names:
>         return None
>     col = names[-1]
>     if len(names) >= 2:
>         qual = names[-2]
>         hits = [qual] if qual in tables else [a for a, t in tables.items() if t["table"] == qual]
>     else:
>         hits = [a for a, t in tables.items() if col in t["cols"]]
>     return (hits[0], col) if len(hits) == 1 else None
>
>
> # ---------- check 1: filters on nullable columns ----------
>
> def _null_contexts(tree, aware=False, soft=False):
>     """Yield (column_ref, null_aware, soft) for every column in a WHERE tree.
>     null_aware: inside IS [NOT] NULL, COALESCE/IFNULL, IS [NOT] DISTINCT FROM.
>     soft:       under OR / CASE, where a NULL might not decide the row."""
>     if isinstance(tree, list):
>         for item in tree:
>             yield from _null_contexts(item, aware, soft)
>         return
>     if not isinstance(tree, dict):
>         return
>     if "class" in tree:
>         if tree["class"] == "SUBQUERY":
>             return
>         if tree["class"] == "COLUMN_REF":
>             yield tree, aware, soft
>             return
>         if (tree.get("type") in _NULL_AWARE_TYPES
>                 or str(tree.get("function_name", "")).lower() in _NULL_AWARE_FUNCS):
>             aware = True
>         if tree.get("type") == "CONJUNCTION_OR" or tree["class"] == "CASE":
>             soft = True
>     for value in tree.values():
>         if isinstance(value, (dict, list)):
>             yield from _null_contexts(value, aware, soft)
>
>
> # WHY A FILTER ON A NULLABLE COLUMN IS A CORRECTNESS ISSUE, NOT A STYLE ISSUE
> #
> # SQL predicates have three outcomes: TRUE, FALSE, UNKNOWN. Comparing anything
> # with NULL gives UNKNOWN, so for a customer whose region is NULL:
> #     region =  'West'        -> UNKNOWN
> #     region <> 'West'        -> UNKNOWN
> #     NOT (region = 'West')   -> UNKNOWN    (NOT UNKNOWN is still UNKNOWN)
> # WHERE keeps only TRUE rows, so that customer lands in neither "West" nor
> # "not West". There's no error and no row-count hint, just a normal-looking number.
> #
> # A style issue is one where a rewrite changes how the query reads but not what
> # it returns. This one changes what the number means: "average revenue of West
> # customers" quietly becomes "average revenue of customers whose region was
> # recorded as West". Segments stop adding up to the total. When the NULLs
> # aren't random (yours are customers from before region was mandatory, an
> # older cohort), the result can be biased, not just based on fewer rows. The
> # same rule makes `x NOT IN (subquery)` return nothing if the subquery yields
> # a single NULL.
> #
> # The reviewer can't know whether excluding unknowns was intended, so it only
> # asks for the intent to be written down: IS [NOT] NULL, COALESCE/IFNULL or
> # IS [NOT] DISTINCT FROM on the column counts as handled. Severity comes from
> # the data: high if NULLs exist now, low if the column is only nullable.
> def _check_null_filters(con, where, tables):
>     found = [(_resolve(ref, tables), aware, soft) for ref, aware, soft in _null_contexts(where)]
>     handled = {key for key, aware, _ in found if key and aware}
>     flagged = {}   # (alias, column) -> True if the NULL rows are certainly dropped
>     for key, aware, soft in found:
>         if key and not aware and key not in handled:
>             flagged[key] = flagged.get(key, False) or not soft
>
>     out = []
>     for (alias, col), certain in flagged.items():
>         info = tables[alias]
>         if not info["cols"].get(col, True):
>             continue   # declared NOT NULL / primary key: nothing unknown to drop
>         n_null, n_total = con.execute(
>             f"SELECT COUNT(*) - COUNT({_q(col)}), COUNT(*) FROM {info['sql']}").fetchone()
>         share = n_null / n_total if n_total else 0.0
>         target = f"{info['table']}.{col}"
>         if n_null == 0:
>             severity = "low"
>             message = (f"WHERE filters on nullable {target}, which has no NULLs today; "
>                        f"the first one that arrives will be dropped without an error.")
>         else:
>             severity = "high" if certain else "medium"
>             message = (f"WHERE filters on nullable {target}: {n_null:,} of {n_total:,} rows "
>                        f"({share:.1%}) are NULL, evaluate to UNKNOWN, and are "
>                        f"{'silently excluded' if certain else 'possibly excluded'}.")
>         out.append(QueryWarning(NULLABLE_FILTER_DROPS_UNKNOWN, severity, message, {
>             "table": info["table"], "column": col, "alias": alias,
>             "null_rows": int(n_null), "total_rows": int(n_total),
>             "null_share": round(share, 4), "drop_is_certain": certain,
>             "fix": (f"state the intent: '{alias}.{col} IS NOT NULL AND ...' if unknowns are "
>                     f"meant to be excluded; COALESCE({alias}.{col}, <default>) or "
>                     f"'... OR {alias}.{col} IS NULL' if they are not."),
>         }))
>     return out
>
>
> # ---------- check 2: joins that change row count ----------
>
> def _join_keys(ref, left, right, tables):
>     """[(left_alias, left_col, right_alias, right_col), ...] for a pure equi-join, else None."""
>     pairs = []
>     if ref.get("using_columns"):
>         for col in (c.lower() for c in ref["using_columns"]):
>             l = [a for a in left if a in tables and col in tables[a]["cols"]]
>             r = [a for a in right if a in tables and col in tables[a]["cols"]]
>             if len(l) != 1 or len(r) != 1:
>                 return None
>             pairs.append((l[0], col, r[0], col))
>     elif ref.get("condition"):
>         for cond in _conjuncts(ref["condition"]):
>             if cond.get("type") != "COMPARE_EQUAL":
>                 return None
>             a = _resolve(cond["left"], tables) if cond["left"].get("class") == "COLUMN_REF" else None
>             b = _resolve(cond["right"], tables) if cond["right"].get("class") == "COLUMN_REF" else None
>             if a is None or b is None:
>                 return None
>             if a[0] in left and b[0] in right:
>                 pairs.append(a + b)
>             elif b[0] in left and a[0] in right:
>                 pairs.append(b + a)
>             else:
>                 return None
>     else:
>         return None   # cross / comma / natural join
>     # The histogram trick needs each side's keys to come from one table.
>     if len({p[0] for p in pairs}) != 1 or len({p[2] for p in pairs}) != 1:
>         return None
>     return pairs
>
>
> # HOW FAN-OUT IS DETECTED WITHOUT RUNNING THE JOIN (LET ALONE TWICE)
> #
> # The obvious check runs the query, then runs it again without the join (or
> # with COUNT(DISTINCT customer_id) beside COUNT(*)) and compares. That runs
> # the expensive part twice, and at scale the fanned-out join *is* the
> # expensive part.
> #
> # For an equi-join, the output size depends only on how many rows each side
> # has per key value:
> #
> #     rows(L JOIN R ON L.k = R.k) = SUM over k of n_L(k) * n_R(k)
> #
> # So each side is collapsed to a histogram (one row per key, with its count),
> # the two histograms are joined (one row per distinct key: about 5,000 here,
> # not one per order), and everything is read off that small result:
> #     inner-join rows   SUM(n_L * n_R)          exact, not an optimizer estimate
> #     fan-out           MAX(n_R) > 1 on a matched key -> some left rows repeat
> #     rows repeated     SUM(n_L) where n_R > 1
> #     orphans           left rows whose key never appears on the right
> #                       (includes NULL keys: NULL = NULL is UNKNOWN, never a match)
> # Single-table WHERE conjuncts are pushed into the histograms with
> # COUNT(*) FILTER (WHERE ...). One scan per table then yields both the filtered
> # count (what your query sees, so rows_before/rows_after match its n_rows)
> # and the unfiltered count (which separates "dropped by your filter" from
> # "dropped because the key has no partner").
> # Cost: one GROUP BY per table and a key-level join. The row-level join never runs.
> def _profile_join(con, pairs, tables, filters):
>     def histogram(alias, cols):
>         keys = ", ".join(f"{_q(alias)}.{_q(c)} AS k{i}" for i, c in enumerate(cols))
>         cond = " AND ".join(filters.get(alias, [])) or "TRUE"
>         return (f"SELECT {keys}, COUNT(*) AS n_all, COUNT(*) FILTER (WHERE {cond}) AS n "
>                 f"FROM {tables[alias]['sql']} AS {_q(alias)} GROUP BY ALL")
>
>     idx = range(len(pairs))
>     on = " AND ".join(f"l.k{i} = r.k{i}" for i in idx)
>     null_key = " OR ".join(f"k{i} IS NULL" for i in idx)
>     sql = f"""
>         WITH l AS ({histogram(pairs[0][0], [p[1] for p in pairs])}),
>              r AS ({histogram(pairs[0][2], [p[3] for p in pairs])}),
>              m AS (SELECT l.n AS ln, r.n AS rn FROM l JOIN r ON {on})
>         SELECT * FROM
>           (SELECT SUM(n) AS left_rows,
>                   SUM(n) FILTER (WHERE {null_key}) AS left_null_key_rows FROM l) AS ls,
>           (SELECT SUM(n) AS right_rows,
>                   SUM(n) FILTER (WHERE {null_key}) AS right_null_key_rows FROM r) AS rs,
>           (SELECT SUM(ln * rn)                  AS inner_join_rows,
>                   SUM(ln)                       AS left_rows_key_exists,
>                   SUM(rn)                       AS right_rows_key_exists,
>                   SUM(ln) FILTER (WHERE rn > 0) AS left_rows_matched,
>                   SUM(rn) FILTER (WHERE ln > 0) AS right_rows_matched,
>                   SUM(ln) FILTER (WHERE rn > 1) AS left_rows_repeated,
>                   SUM(rn) FILTER (WHERE ln > 1) AS right_rows_repeated,
>                   MAX(rn) FILTER (WHERE ln > 0) AS max_matches_per_left_row,
>                   MAX(ln) FILTER (WHERE rn > 0) AS max_matches_per_right_row
>            FROM m) AS ms
>     """
>     row = con.execute(sql).df().fillna(0).iloc[0]
>     return {name: int(value) for name, value in row.items()}
>
>
> def _check_joins(con, where, tables, joins):
>     """Profile each equi-join. Returns (warnings, fanned), where fanned maps
>     alias -> joins that repeat that alias's rows."""
>     # WHERE can't be pushed into the NULL-filling side of an outer join.
>     null_side = set()
>     for j in joins:
>         jt = j["ref"].get("join_type")
>         if jt == "LEFT":
>             null_side |= j["right"]
>         elif jt == "RIGHT":
>             null_side |= j["left"]
>         elif jt in ("OUTER", "FULL"):
>             null_side |= j["left"] | j["right"]
>
>     filters = {}
>     for conj in _conjuncts(where):
>         keys = [_resolve(c, tables) for c in _columns(conj)]
>         aliases = {k[0] for k in keys if k}
>         if (not keys or None in keys or len(aliases) != 1 or aliases & null_side
>                 or any(e["class"] == "SUBQUERY" for e in _exprs(conj))):
>             continue
>         try:
>             filters.setdefault(aliases.pop(), []).append(_render(con, conj))
>         except Exception:
>             pass   # can't deparse: that side is profiled unfiltered
>
>     out, fanned = [], {}
>     for j in joins:
>         ref = j["ref"]
>         jt = ref.get("join_type") or "INNER"
>         pairs = _join_keys(ref, j["left"], j["right"], tables)
>         if pairs is None:
>             out.append(QueryWarning(REVIEW_INCOMPLETE, "info",
>                 f"{jt} join without a plain column = column condition (cross, comma, natural "
>                 f"or non-equi join) was not profiled; its effect on row count is unknown.",
>                 {"join_type": jt}))
>             continue
>         label = " AND ".join(f"{a}.{x} = {b}.{y}" for a, x, b, y in pairs)
>         try:
>             s = _profile_join(con, pairs, tables, filters)
>         except Exception as e:
>             out.append(QueryWarning(REVIEW_INCOMPLETE, "info",
>                                     f"could not profile join {label}: {e}"))
>             continue
>
>         left_unmatched = s["left_rows"] - s["left_rows_matched"]
>         right_unmatched = s["right_rows"] - s["right_rows_matched"]
>         rows_after = {
>             "INNER": s["inner_join_rows"],
>             "LEFT":  s["inner_join_rows"] + left_unmatched,
>             "RIGHT": s["inner_join_rows"] + right_unmatched,
>             "OUTER": s["inner_join_rows"] + left_unmatched + right_unmatched,
>             "FULL":  s["inner_join_rows"] + left_unmatched + right_unmatched,
>             "SEMI":  s["left_rows_matched"],
>             "ANTI":  left_unmatched,
>         }.get(jt, s["inner_join_rows"])
>         multiplies = jt not in ("SEMI", "ANTI")
>
>         # Record whose rows get repeated, and by how much (used by check 3).
>         if multiplies:
>             for side in ("left", "right"):
>                 most = s[f"max_matches_per_{side}_row"]
>                 if most > 1:
>                     for alias in j[side]:
>                         fanned.setdefault(alias, []).append({
>                             "join": label, "max_copies": most,
>                             "avg_copies": round(s["inner_join_rows"] / s[f"{side}_rows_matched"], 2)})
>
>         repeated = s["left_rows_repeated"] if multiplies else 0
>         orphans = (s["left_rows"] - s["left_rows_key_exists"]) if jt in ("INNER", "RIGHT", "SEMI") else 0
>         if repeated or orphans:
>             lt, rt = tables[pairs[0][0]]["table"], tables[pairs[0][2]]["table"]
>             reasons = []
>             if repeated:
>                 reasons.append(f"{repeated:,} {lt} rows match more than one {rt} row "
>                                f"(up to {s['max_matches_per_left_row']:,})")
>             if orphans:
>                 reasons.append(f"{orphans:,} {lt} rows have no {rt} row at all "
>                                f"({s['left_null_key_rows']:,} with a NULL key) and vanish")
>             out.append(QueryWarning(JOIN_ROW_COUNT_CHANGE, "medium",
>                 f"{jt} JOIN on {label} turns {s['left_rows']:,} {lt} rows into "
>                 f"{rows_after:,}: " + "; ".join(reasons) + ".",
>                 {"join": label, "join_type": jt, "left_table": lt, "right_table": rt,
>                  "rows_before": s["left_rows"], "rows_after": rows_after,
>                  "ratio": round(rows_after / s["left_rows"], 3) if s["left_rows"] else None,
>                  "filters_applied": {a: filters.get(a, []) for a in (pairs[0][0], pairs[0][2])},
>                  **s}))
>     return out, fanned
>
>
> # ---------- check 3: aggregates over a fanned-out result ----------
>
> def _check_aggregates(con, node, tables, fanned, driving):
>     agg_names = {r[0].lower() for r in con.execute(
>         "SELECT DISTINCT function_name FROM duckdb_functions() "
>         "WHERE function_type = 'aggregate'").fetchall()} | {"count_star"}
>     out = []
>     for clause in ("select_list", "having"):
>         for agg in _exprs(node.get(clause)):
>             name = str(agg.get("function_name", "")).lower()
>             if agg["class"] != "FUNCTION" or name not in agg_names:
>                 continue
>             if agg.get("distinct") or name in _DUP_INSENSITIVE:
>                 continue   # MIN, MAX, COUNT(DISTINCT ...) are unchanged by repeated rows
>             keys = [_resolve(c, tables) for c in _columns(agg.get("children"))]
>             # COUNT(*) / SUM(1) read no column: they count rows of the driving table.
>             aliases = {k[0] for k in keys if k} or ({driving} if driving else set())
>             # SUM(o.qty * c.price) is fine while o is at its own grain, so flag
>             # only when every table the aggregate reads from has been fanned out.
>             if not aliases or not all(a in fanned for a in aliases):
>                 continue
>             joins = [e for a in sorted(aliases) for e in fanned[a]]
>             worst = max(joins, key=lambda e: e["max_copies"])
>             fanned_tables = sorted({tables[a]["table"] for a in aliases if a in tables})
>             if name == "count_star" or not agg.get("children"):
>                 shown = "COUNT(*)"
>             else:
>                 try:
>                     shown = _render(con, agg)
>                 except Exception:
>                     shown = name
>             if name in ("avg", "mean"):
>                 effect = "is weighted by how many times each row was repeated"
>             elif name in ("sum", "count", "count_star"):
>                 effect = "is inflated"
>             else:
>                 effect = "is computed over repeated rows"
>             out.append(QueryWarning(AGGREGATE_OVER_FANOUT, "high",
>                 f"{shown} {effect}: {', '.join(fanned_tables)} rows are repeated by the join "
>                 f"on {worst['join']} (avg {worst['avg_copies']}x, max {worst['max_copies']:,}x).",
>                 {"aggregate": name, "expression": shown,
>                  "output_column": agg.get("alias") or None,
>                  "fanned_tables": fanned_tables, "joins": joins,
>                  "fix": ("compute it at the intended grain: pre-aggregate the other table to one "
>                          "row per join key, or use EXISTS if the join is only a filter.")}))
>     return out
>
>
> # ---------- entry point ----------
>
> def review_query(con: duckdb.DuckDBPyConnection, sql: str) -> list[QueryWarning]:
>     """Review one SELECT against the live data in `con`. Returns QueryWarning objects."""
>     try:
>         parsed = _parse(con, sql)
>     except Exception as e:
>         return [QueryWarning(REVIEW_INCOMPLETE, "info", f"could not parse query: {e}")]
>     statements = parsed.get("statements") or []
>     if parsed.get("error") or len(statements) != 1:
>         return [QueryWarning(REVIEW_INCOMPLETE, "info",
>                              parsed.get("error_message") or "expected exactly one SELECT")]
>     node = statements[0]["node"]
>     if node.get("type") != "SELECT_NODE":
>         return [QueryWarning(REVIEW_INCOMPLETE, "info",
>                              f"{node.get('type')} (e.g. UNION) is not analysed")]
>
>     tables, joins, notes = {}, [], []
>     from_table = node.get("from_table") or {}
>     _flatten_from(from_table, _schema(con), tables, joins, notes)
>     if (node.get("cte_map") or {}).get("map"):
>         notes.append("WITH clause present; CTE bodies are not analysed")
>     warnings = [QueryWarning(REVIEW_INCOMPLETE, "info", n) for n in notes]
>
>     where = node.get("where_clause")
>     warnings += _check_null_filters(con, where, tables)
>     join_warnings, fanned = _check_joins(con, where, tables, joins)
>     warnings += join_warnings
>     warnings += _check_aggregates(con, node, tables, fanned, _leftmost_alias(from_table))
>     return warnings
> ```
>
> ## Cell 2: run on the Phase 1 queries, plus tests
>
> ```python
> FINANCE_SQL = """
> SELECT AVG(c.revenue) AS avg_revenue,
>        COUNT(*)       AS n_rows
> FROM customers AS c
> WHERE c.region = 'West'
> """
>
> DASHBOARD_SQL = """
> SELECT AVG(c.revenue) AS avg_revenue,
>        COUNT(*)       AS n_rows
> FROM customers AS c
> JOIN orders AS o
>   ON o.customer_id = c.customer_id
> WHERE c.region = 'West'
> """
>
> fin = review_query(con, FINANCE_SQL)
> dash = review_query(con, DASHBOARD_SQL)
> display(warnings_frame(fin))
> display(warnings_frame(dash))
>
> def codes(ws):
>     return sorted(w.code for w in ws)
>
> # 1. Finance: only the NULL-region warning.
> assert codes(fin) == [NULLABLE_FILTER_DROPS_UNKNOWN], codes(fin)
>
> # 2. Dashboard: NULL warning, row-count change, and fan-out on both AVG and COUNT(*).
> assert codes(dash) == sorted([NULLABLE_FILTER_DROPS_UNKNOWN, JOIN_ROW_COUNT_CHANGE,
>                               AGGREGATE_OVER_FANOUT, AGGREGATE_OVER_FANOUT]), codes(dash)
>
> # 3. The histogram counts must reconcile exactly with each query's own n_rows.
> join_w = next(w for w in dash if w.code == JOIN_ROW_COUNT_CHANGE)
> assert join_w.details["rows_before"] == con.execute(FINANCE_SQL).fetchone()[1]
> assert join_w.details["rows_after"] == con.execute(DASHBOARD_SQL).fetchone()[1]
>
> # 4. Same bug written from the orders side: the orders row count is preserved,
> #    but customer rows still repeat, so the aggregate check must still fire.
> FLIPPED_SQL = """
> SELECT AVG(c.revenue) FROM orders AS o
> JOIN customers AS c ON c.customer_id = o.customer_id
> WHERE c.region = 'West'
> """
> assert AGGREGATE_OVER_FANOUT in codes(review_query(con, FLIPPED_SQL))
>
> # 5. The fixes clear their warnings.
> EXISTS_SQL = """
> SELECT AVG(c.revenue) AS avg_revenue, COUNT(*) AS n_rows
> FROM customers AS c
> WHERE c.region = 'West'
>   AND EXISTS (SELECT 1 FROM orders AS o WHERE o.customer_id = c.customer_id)
> """
> assert codes(review_query(con, EXISTS_SQL)) == [NULLABLE_FILTER_DROPS_UNKNOWN]
>
> EXPLICIT_SQL = """
> SELECT AVG(c.revenue) AS avg_revenue, COUNT(*) AS n_rows
> FROM customers AS c
> WHERE c.region IS NOT NULL AND c.region = 'West'
> """
> assert review_query(con, EXPLICIT_SQL) == []
> print("all checks passed")
> ```
>
> Test 4 shows why a row-count check isn't enough. Relative to `orders`, nothing changes, so `JOIN_ROW_COUNT_CHANGE` stays quiet. Only the per-table aggregate check catches it. (The join warning will also fire there if some orders have no matching customer, which you'd want to know anyway.)
>
> ## Evaluate
>
> **Expected results:**
>
> | Warning | Finance (~\$812) | Dashboard (~\$1,240) |
> |---|---|---|
> | `NULLABLE_FILTER_DROPS_UNKNOWN` (high) | Fires: about 750 of 5,000 customers (about 15%) have NULL region | Fires, identical |
> | `JOIN_ROW_COUNT_CHANGE` (medium) | No | Fires: `rows_before` equals Finance `n_rows`, `rows_after` equals Dashboard `n_rows` (about 7x if West customers order like everyone else) |
> | `AGGREGATE_OVER_FANOUT` (high) | No | Fires twice: `avg(c.revenue)` and `COUNT(*)` |
>
> **Would it have caught the \$1,240? Yes.** The only difference between your two queries is the join, and that is exactly what the fan-out warning flags. It doesn't need to know that \$812 exists:
>
> - **What the dashboard AVG really computes:** it gives each order one vote instead of each customer: Σ revenueᵢ·kᵢ / Σ kᵢ, where kᵢ is customer i's order count.
> - **Why it comes out higher:** that weighted average goes up whenever customers who order a lot also have higher revenue.
> - **Size of the gap:** it is exactly Cov(revenue, k) / mean(k).
>
> Cell 3 rebuilds both numbers from one query that can't fan out. `dashboard_avg - finance_avg` will equal `gap_from_covariance`, because that's an identity. If the correlation comes out positive, the fan-out accounts for the whole \$428 difference.
>
> ```python
> # Cell 3: does fan-out explain all of the gap?
> con.execute("""
>     WITH k AS (SELECT customer_id, COUNT(*) AS n_orders FROM orders GROUP BY customer_id)
>     SELECT AVG(c.revenue)                                     AS finance_avg,    -- one vote per customer
>            SUM(c.revenue * k.n_orders) / SUM(k.n_orders)      AS dashboard_avg,  -- one vote per order
>            COVAR_POP(c.revenue, k.n_orders) / AVG(k.n_orders) AS gap_from_covariance,
>            CORR(c.revenue, k.n_orders)                        AS corr_revenue_orders
>     FROM customers AS c
>     JOIN k USING (customer_id)        -- k has one row per customer, so no fan-out
>     WHERE c.region = 'West'
> """).df()
> ```
>
> ```python
> # Cell 4: are the NULL-region customers a random sample?
> con.execute("""
>     SELECT region IS NULL AS region_unknown, COUNT(*) AS customers, AVG(revenue) AS avg_revenue,
>            MIN(signup_date) AS first_signup, MAX(signup_date) AS last_signup
>     FROM customers GROUP BY ALL ORDER BY 1
> """).df()
> ```
>
> There are two caveats:
>
> 1. **It only helps if it's a gate.** A list of warnings nobody checks reaches the board just as easily. Make the publish step `assert` that there are no `high` warnings. There was also a free clue already: the dashboard's `n_rows` was about 7 times Finance's.
> 2. **That gate would block the Finance query too, and it should.** The NULL filter doesn't explain the gap, since both queries use the same WHERE. But it does mean \$812 isn't "West average revenue" either. It's the average for customers *recorded* as West. The roughly 750 customers from before region was mandatory are in no region's number, and as an older cohort they probably aren't a random sample (Cell 4 checks this). The fix is one of two things:
>    - say it explicitly: `region IS NOT NULL`, which clears the warning, plus a coverage note on the slide;
>    - or report an "Unknown" bucket.
>
> **Known limits:**
> - **Single plain SELECT only.** CTEs, subqueries in FROM and UNIONs come back as `REVIEW_INCOMPLETE`, never as a clean pass. So the "join to pre-aggregated orders" fix gets "incomplete"; EXISTS is the fix it can confirm.
> - **Chains of joins are profiled one join at a time.** Fan-out stacked across several joins is flagged, but the multipliers aren't combined.
> - **Some joins aren't profiled:** non-equality joins and comma joins are reported as incomplete.
> - **One known false positive:** an AVG that is grouped at the repeated table's grain (for example `GROUP BY c.customer_id`) is safe but still gets flagged.
> - **Writing `IS NOT NULL` silences the NULL warning.** The check makes sure the exclusion is written down, not that it's what you wanted.
> - **Not covered yet:** a WHERE on the right-hand table of a LEFT JOIN, which silently turns it into an inner join. That's the natural next check and would use the same parsed query.

**Changes I made, one line each:**
1. **Cell 1, `_flatten_from` (the only code change):** a subquery in FROM is now profiled as a derived table instead of being skipped, so the correct fix (orders rolled up to one row per customer, then joined) is checked rather than returned as "not analysed". The lines are marked `# CHANGED:` in the Task 4.1 cell.
2. **Cell 2, the model's tests:** used unchanged, in their own cell, to check its predictions on my data.
3. **Cells 3 and 4:** not used. They repeat Step 1.2's weighted-average rebuild and Step 2.1's who-is-missing table, both already in this notebook.
4. **The scored run on my queries is mine, not the model's:** the cell after the Task 4.1 code, with three probe queries the model never saw.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.